# Reproducción de los resultados del test final 2024

Este notebook reproduce las métricas reportadas en la tesis cargando
los pesos de los modelos entrenados (`tft_final_test_2024.ckpt` y
`lstm_final_test_2024.ckpt`). Los modelos NAIVE y ARIMA son
deterministas y se recalculan desde cero.

**Requisitos:**
- `data/final_dataset.csv` (dataset completo, descargable desde Kaggle).
  Las cifras de la tesis fueron generadas sobre los 376 municipios; el
  sample de 50 municipios da resultados distintos porque los embeddings
  del TFT dependen del número de municipios.
- `checkpoints/tft_final_test_2024.ckpt`
- `checkpoints/lstm_final_test_2024.ckpt`

**Salidas:**
- `results/reproduction_2024/<modelo>/`: predicciones y métricas por modelo.
- `results/reproduction_2024/reproduction_summary.xlsx`: resumen consolidado.

Ejecutar las celdas en orden. La inferencia es determinista (modo
evaluación, sin dropout, sin shuffle), por lo que los números deben
coincidir bit-a-bit con los del paper si se usa el mismo dataset y los
checkpoints incluidos en el repositorio.


In [ ]:
import pandas as pd
from pathlib import Path

# ============================================================
# CONFIGURACIÓN DEL NOTEBOOK DE REPRODUCCIÓN
# ============================================================
# Raíz del repositorio (notebook está en notebooks/, así que subimos un nivel).
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

# Por defecto se usa el dataset completo. Las cifras del paper requieren
# el dataset completo de 376 municipios; el sample de 50 municipios sirve
# para verificar que el notebook se ejecuta de extremo a extremo, pero no
# reproduce las cifras de la tesis (los embeddings del TFT no son
# compatibles entre datasets de distinto número de municipios).
DATA_PATH = REPO_ROOT / "data" / "final_dataset.csv"

# Si no existe el dataset completo, caer al sample con un aviso explícito.
if not DATA_PATH.exists():
    sample_path = REPO_ROOT / "data" / "final_dataset_sample.csv"
    if sample_path.exists():
        print(
            "AVISO: no se encontró final_dataset.csv. Cargando el sample de 50 "
            "municipios. Las métricas obtenidas NO coincidirán con las de la "
            "tesis; descargar el dataset completo desde Kaggle para reproducir."
        )
        DATA_PATH = sample_path
    else:
        raise FileNotFoundError(
            f"No se encontró ni final_dataset.csv ni final_dataset_sample.csv en {REPO_ROOT / 'data'}"
        )

# Carpeta donde se guardan los artefactos de la reproducción.
RESULTS_DIR = REPO_ROOT / "results" / "reproduction_2024"

# Rutas a los checkpoints incluidos en el repositorio.
CKPT_DIR = REPO_ROOT / "checkpoints"
TFT_CKPT = CKPT_DIR / "tft_final_test_2024.ckpt"
LSTM_CKPT = CKPT_DIR / "lstm_final_test_2024.ckpt"

assert TFT_CKPT.exists(), f"No se encontró el checkpoint del TFT: {TFT_CKPT}"
assert LSTM_CKPT.exists(), f"No se encontró el checkpoint del LSTM: {LSTM_CKPT}"

print(f"Dataset:        {DATA_PATH}")
print(f"Checkpoint TFT: {TFT_CKPT}")
print(f"Checkpoint LSTM:{LSTM_CKPT}")
print(f"Salidas en:     {RESULTS_DIR}")

df = pd.read_csv(DATA_PATH)


## Definiciones del pipeline

La siguiente celda contiene todas las definiciones de funciones, clases y
constantes del pipeline. No entrena ningún modelo: se limita a dejar las
herramientas disponibles para que las celdas siguientes carguen los
checkpoints y calculen métricas.


In [ ]:
# =========================
# 0. IMPORTS
# =========================
import os
import gc
import json
import math
import random
import shutil
import warnings
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict, List, Tuple, Optional, Any

from tqdm.auto import tqdm

from scipy import stats
from statsmodels.stats.contingency_tables import mcnemar

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

try:
    torch.set_float32_matmul_precision("medium")
except Exception:
    pass

try:
    import lightning.pytorch as pl
    from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint, LearningRateMonitor, Callback
    from lightning.pytorch.loggers import CSVLogger
except ImportError:
    import pytorch_lightning as pl
    from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint, LearningRateMonitor, Callback
    from pytorch_lightning.loggers import CSVLogger

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, precision_recall_fscore_support

# TFT imports
from pytorch_forecasting import TimeSeriesDataSet, TemporalFusionTransformer
from pytorch_forecasting.data import GroupNormalizer
try:
    from pytorch_forecasting.metrics import NegativeBinomialDistributionLoss
except Exception:
    from pytorch_forecasting.metrics.distributions import NegativeBinomialDistributionLoss

# Classical time-series imports
try:
    from pmdarima import auto_arima
    PMDARIMA_AVAILABLE = True
except Exception:
    PMDARIMA_AVAILABLE = False

from statsmodels.tsa.arima.model import ARIMA

warnings.filterwarnings("ignore")

# =========================
# 1. CONFIGURATION
# =========================
SEED = 42
pl.seed_everything(SEED, workers=True)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.empty_cache()

# Determinismo reforzado. Estos flags eliminan las fuentes habituales de
# variabilidad entre corridas en CPU. En GPU pueden persistir pequeñas
# diferencias residuales en kernels sin contraparte determinista.
os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
try:
    torch.use_deterministic_algorithms(True, warn_only=True)
except Exception as _e:
    print(f"[determinismo] aviso: {_e}")


def reseed_for_run(tag: str, base_seed: int = SEED) -> int:
    """
    Re-siembra todos los RNGs antes de entrenar o evaluar un modelo.

    Genera un seed determinista a partir de un tag de texto, de forma que
    el resultado de cada (fold, modelo) sea independiente del orden o la
    cantidad de trabajo aleatorio realizado previamente en la sesión.
    Esto evita que un cambio en el estado del RNG global (por ejemplo, al
    saltarse pasos por reanudación) altere los resultados.
    """
    import hashlib
    h = hashlib.sha256(f"{base_seed}:{tag}".encode()).digest()
    derived = int.from_bytes(h[:4], "big") % (2**31 - 1)

    random.seed(derived)
    np.random.seed(derived)
    torch.manual_seed(derived)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(derived)
    pl.seed_everything(derived, workers=True)
    return derived


RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Core columns
GROUP_COL = "COD_MUN_N"
TIME_COL = "time_idx"
YEAR_COL = "ANO"
TARGET_COL = "casos_totales"
FLOW_COL = "Flujo_in"
AGE_COLS = ["casos_0_4", "casos_5_14", "casos_15_64", "casos_65_plus"]
CLIMATE_COLS = ["prec_total", "ndvi_mean", "oni_anom"]
IMPUTATION_FLAG_COLS = []

# Global modeling window
MAX_ENCODER_LENGTH = 26
MIN_ENCODER_LENGTH = 13
MAX_PREDICTION_LENGTH = 4

# Fold years
VALIDATION_YEARS = [2021, 2022, 2023]
TEST_YEAR = 2024

# Threshold de clasificación positivo / cero
POSITIVE_CLASSIFICATION_THRESHOLD = 0.9

# Evaluación relajada de brote: una semana cuenta como "brote relajado" si
# alcanza el 70% del umbral estricto.
OUTBREAK_RELAXED_FACTOR = 0.7

# =========================
# DEFINICIÓN DE BROTE (DINÁMICA, SIN FUGA, SIN CLASE EXTREMA)
# =========================
# Brote = aumento abrupto respecto al baseline local reciente.
OUTBREAK_BASELINE_WINDOW = 4
OUTBREAK_BASELINE_MIN_HISTORY = 2
OUTBREAK_GROWTH_RATE = 0.75
OUTBREAK_MIN_CASES_ABS = 3.0
OUTBREAK_BASELINE_FLOOR = 0.0

# Pesos por defecto. Pueden ser sobrescritos por el tuning de pesos.
LOSS_W_ZERO = 1.0
LOSS_W_POSITIVE_NON_OUTBREAK = 1.5
LOSS_W_OUTBREAK = 6.25

# Búsqueda de pesos para el TFT antes del test final.
TUNE_TFT_HYPERPARAMS = True
TUNE_TFT_FOLD_YEAR = 2023
TUNE_TFT_MAX_EPOCHS = 10
TUNE_SUBSAMPLE_FRACTION = 0.5

# Filtro opcional de municipios
USE_ACTIVE_MUNICIPALITIES = False
ACTIVE_MIN_TOTAL_CASES = 25
ACTIVE_MIN_POSITIVE_RATE = 0.02
ACTIVE_MIN_MAX_CASES = 5

# Lags / rolls (vacíos por diseño; el TFT los aprende internamente)
TARGET_LAGS = []
FLOW_LAGS = []
AGE_LAGS = []
TARGET_ROLLS = []
FLOW_ROLLS = []
AGE_ROLLS = []

# =========================
# TFT: hiperparámetros
# =========================
# Arquitectura fija reportada en la tesis. Se usa idéntica durante la CV
# ligera (selección de mediana de épocas) y durante el test final. Los
# pesos de pérdida pueden ser sobrescritos por el tuning sobre el fold
# TUNE_TFT_FOLD_YEAR; la arquitectura permanece fija.
TFT_BATCH_SIZE = 64
TFT_MAX_EPOCHS = 50
TFT_LEARNING_RATE = 0.003
TFT_GRAD_CLIP = 0.1
TFT_HIDDEN_SIZE = 64
TFT_ATTENTION_HEAD_SIZE = 4
TFT_DROPOUT = 0.15
TFT_HIDDEN_CONT_SIZE = 32
TOP_K_CKPTS = 4

# LSTM
LSTM_BATCH_SIZE = 64
LSTM_MAX_EPOCHS = 50
LSTM_LEARNING_RATE = 0.003
LSTM_GRAD_CLIP = 0.5
LSTM_HIDDEN_SIZE = 64
LSTM_NUM_LAYERS = 2
LSTM_DROPOUT = 0.15
LSTM_STATIC_HIDDEN = 32
LSTM_EMBED_DIM = 16

# ARIMA
ARIMA_MAX_HISTORY = 156
ARIMA_USE_AUTO_ARIMA = True

# Misc
SAVE_METRICS_BY_MUNICIPALITY = False
SAVE_PREDICTIONS_FULL_PARQUET = True
SAVE_PREDICTIONS_SAMPLE = True
PREDICTIONS_SAMPLE_N = 5000
FUTURE_CLIMATE_IS_TRULY_KNOWN = False
REFIT_FINAL_PRETEST = True
USE_GPU_IF_AVAILABLE = True
NUM_WORKERS = 0

# Modelos a correr en el test final
RUN_MODELS = ["NAIVE", "ARIMA", "LSTM", "TFT"]
ALLOW_RESUME = True
FORCE_RETRAIN_COMPLETED_MODELS = False
FORCE_REBUILD_FOLD_COMPARISON = True

SHEET_ORDER = [
    "summary_overall",
    "regression_by_horizon",
    "positive_by_horizon",
    "outbreak_by_horizon",
    "outbreak_confusion_by_horizon",
    "anticipation_summary",
    "anticipation_event_level",
    "error_strata",
    "predictions_sample",
    "training_history",
    "top_checkpoints",
    "metrics_by_municipality",
    "predictions_metadata",
]

REGRESSION_COLUMNS_ORDER = [
    "model_name", "fold_name", "partition_name", "horizon", "n_obs",
    "true_zero_rate", "pred_exact_zero_rate", "pred_below_positive_threshold_rate",
    "MAE", "RMSE", "sMAPE", "R2",
    "MAE_nonzero", "RMSE_nonzero", "sMAPE_nonzero",
    "mean_true_when_positive", "mean_pred_when_positive",
    "median_true_when_positive", "median_pred_when_positive",
    "mean_pred_when_true_zero", "median_pred_when_true_zero",
    "pearson_corr", "spearman_corr"
]

POSITIVE_COLUMNS_ORDER = [
    "model_name", "fold_name", "partition_name", "horizon", "n_obs",
    "n_true_positive_obs", "true_positive_rate",
    "model_precision", "model_recall", "model_f1",
    "model_false_alarm_rate", "model_specificity", "model_balanced_accuracy",
    "model_tp", "model_fp", "model_tn", "model_fn"
]

OUTBREAK_COLUMNS_ORDER = [
    "model_name", "fold_name", "partition_name", "horizon", "n_obs",

    # Strict / original outbreak evaluation
    "n_true_outbreaks", "true_outbreak_rate",
    "model_detected_outbreak_obs", "model_missed_outbreak_obs", "model_false_alarm_outbreak_obs",
    "model_precision", "model_recall", "model_f1",
    "model_false_alarm_rate", "model_specificity", "model_balanced_accuracy",

    # Relaxed prediction vs strict truth
    "model_detected_outbreak_obs_relaxed", "model_missed_outbreak_obs_relaxed", "model_false_alarm_outbreak_obs_relaxed",
    "model_precision_relaxed", "model_recall_relaxed", "model_f1_relaxed",
    "model_false_alarm_rate_relaxed", "model_specificity_relaxed", "model_balanced_accuracy_relaxed",

    # Errors on strict true outbreaks only
    "MAE_on_true_outbreaks",
    "RMSE_on_true_outbreaks",
    "sMAPE_on_true_outbreaks",
    "relative_bias_on_outbreaks",
    "underprediction_rate_on_outbreaks", "overprediction_rate_on_outbreaks",
]

SUMMARY_COLUMNS_ORDER = [
    "model_name", "fold_name", "partition_name",
    "positive_threshold", "outbreak_threshold_rule", "outbreak_relaxed_factor",
    "n_obs", "true_zero_rate", "pred_exact_zero_rate", "pred_below_positive_threshold_rate",
    "MAE", "RMSE", "sMAPE", "R2",
    "pearson_corr", "spearman_corr",

    # Positive
    "positive_model_precision", "positive_model_recall", "positive_model_f1",
    "positive_model_false_alarm_rate", "positive_model_specificity", "positive_model_balanced_accuracy",

    # Strict outbreak
    "outbreak_model_precision", "outbreak_model_recall", "outbreak_model_f1",
    "outbreak_model_missed_rate", "outbreak_model_false_alarm_rate",
    "outbreak_model_specificity", "outbreak_model_balanced_accuracy", "outbreak_model_positive_rate",

    # Relaxed prediction vs strict truth
    "outbreak_model_precision_relaxed", "outbreak_model_recall_relaxed", "outbreak_model_f1_relaxed",
    "outbreak_model_missed_rate_relaxed", "outbreak_model_false_alarm_rate_relaxed",
    "outbreak_model_specificity_relaxed", "outbreak_model_balanced_accuracy_relaxed",
    "outbreak_model_positive_rate_relaxed",

    # Errors on strict true outbreaks only
    "MAE_on_true_outbreaks", "RMSE_on_true_outbreaks", "sMAPE_on_true_outbreaks",
    "mean_signed_error_on_outbreaks", "mean_abs_error_on_outbreaks", "median_signed_error_on_outbreaks",
    "mean_true_on_outbreaks", "mean_pred_on_outbreaks",
    "relative_bias_on_outbreaks",
    "underprediction_rate_on_outbreaks", "overprediction_rate_on_outbreaks",
]

# =========================
# 2. DATACLASSES
# =========================
@dataclass
class FoldSpec:
    fold_name: str
    eval_start_year: int
    partition_name: str
    train_df: pd.DataFrame
    eval_df: pd.DataFrame
    output_dir: Path

@dataclass
class ModelArtifacts:
    model_name: str
    fold_name: str
    partition_name: str
    overall: Dict[str, Any]
    regression_by_horizon: pd.DataFrame
    positive_by_horizon: pd.DataFrame
    outbreak_by_horizon: pd.DataFrame
    outbreak_confusion_by_horizon: pd.DataFrame
    anticipation_summary: pd.DataFrame
    anticipation_event_level: pd.DataFrame
    error_strata: pd.DataFrame
    predictions: pd.DataFrame
    training_history: pd.DataFrame
    top_checkpoints: pd.DataFrame
    metrics_by_municipality: Optional[pd.DataFrame]
    metadata: Dict[str, Any]

# =========================
# 3. BASIC UTILITIES
# =========================
def ensure_dir(path: Path):
    path.mkdir(parents=True, exist_ok=True)

def safe_float(x, default=np.nan):
    try:
        if pd.isna(x):
            return default
        return float(x)
    except Exception:
        return default

def get_trainer_accelerator():
    if USE_GPU_IF_AVAILABLE and torch.cuda.is_available():
        return {"accelerator": "gpu", "devices": 1}
    return {"accelerator": "cpu", "devices": 1}

def ensure_string_group(df_in: pd.DataFrame) -> pd.DataFrame:
    out = df_in.copy()
    out[GROUP_COL] = out[GROUP_COL].astype(str)
    return out

def sanitize_sheet_name(name: str) -> str:
    invalid = ['\\', '/', '*', '?', ':', '[', ']']
    out = name
    for ch in invalid:
        out = out.replace(ch, "_")
    return out[:31]

def reorder_columns(df_in: pd.DataFrame, ordered_cols: List[str]) -> pd.DataFrame:
    if df_in is None or len(df_in) == 0:
        return df_in
    existing = [c for c in ordered_cols if c in df_in.columns]
    remaining = [c for c in df_in.columns if c not in existing]
    return df_in[existing + remaining].copy()

def write_json(path: Path, obj: Any):
    ensure_dir(path.parent)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, default=str)

def read_json(path: Path, default=None):
    if not path.exists():
        return default
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def safe_to_parquet(df_in: pd.DataFrame, path: Path):
    ensure_dir(path.parent)
    try:
        df_in.to_parquet(path, index=False)
    except Exception:
        df_in.to_csv(path.with_suffix(".csv"), index=False)

def safe_read_dataframe(path: Path) -> Optional[pd.DataFrame]:
    if path.exists():
        if path.suffix.lower() == ".parquet":
            return pd.read_parquet(path)
        if path.suffix.lower() == ".csv":
            return pd.read_csv(path)
    if path.with_suffix(".csv").exists():
        return pd.read_csv(path.with_suffix(".csv"))
    return None

def model_done_flag(path: Path) -> Path:
    return path / "_DONE.json"

# =========================
# 4. DATA INTEGRITY UTILITIES
# =========================
def check_required_non_nulls(df_in: pd.DataFrame, name: str):
    needed = [GROUP_COL, TIME_COL, YEAR_COL, TARGET_COL]
    null_counts = df_in[needed].isnull().sum().to_dict()
    bad = {k: int(v) for k, v in null_counts.items() if v > 0}
    if bad:
        raise ValueError(f"[{name}] Hay nulos en columnas clave: {bad}")

def validate_prediction_keys_across_models(
    artifacts_dict: Dict[str, ModelArtifacts],
    strict: bool = True,
) -> pd.DataFrame:
    """
    Verifica que todos los modelos tengan exactamente las mismas llaves de evaluación:
    (municipio, prediction_start_time_idx, horizon, target_time_idx).

    Retorna un DataFrame resumen.
    Si strict=True y hay diferencias, lanza ValueError.
    """
    key_cols = [GROUP_COL, "prediction_start_time_idx", "horizon", "target_time_idx"]

    available = {}
    for model_name, art in artifacts_dict.items():
        if art is None or art.predictions is None or len(art.predictions) == 0:
            continue

        tmp = art.predictions[key_cols].drop_duplicates().copy()

        # Forzar tipos consistentes antes de comparar
        tmp[GROUP_COL] = tmp[GROUP_COL].astype(str)
        tmp["prediction_start_time_idx"] = tmp["prediction_start_time_idx"].astype(int)
        tmp["horizon"] = tmp["horizon"].astype(int)
        tmp["target_time_idx"] = tmp["target_time_idx"].astype(int)

        available[model_name] = tmp

    rows = []
    if len(available) <= 1:
        return pd.DataFrame(columns=[
            "reference_model", "compared_model",
            "n_reference_keys", "n_compared_keys",
            "n_missing_in_compared", "n_extra_in_compared",
            "keys_match_exactly"
        ])

    ref_model = sorted(available.keys())[0]
    ref_keys = available[ref_model].copy()

    for model_name, df_keys in available.items():
        tmp_ref = ref_keys[key_cols].drop_duplicates().copy()
        tmp_cmp = df_keys[key_cols].drop_duplicates().copy()

        merged_ref = tmp_ref.merge(tmp_cmp, on=key_cols, how="left", indicator=True)
        missing_in_cmp = int((merged_ref["_merge"] == "left_only").sum())

        merged_cmp = tmp_cmp.merge(tmp_ref, on=key_cols, how="left", indicator=True)
        extra_in_cmp = int((merged_cmp["_merge"] == "left_only").sum())

        match = (missing_in_cmp == 0) and (extra_in_cmp == 0)

        rows.append({
            "reference_model": ref_model,
            "compared_model": model_name,
            "n_reference_keys": int(len(tmp_ref)),
            "n_compared_keys": int(len(tmp_cmp)),
            "n_missing_in_compared": missing_in_cmp,
            "n_extra_in_compared": extra_in_cmp,
            "keys_match_exactly": bool(match),
        })

    report_df = pd.DataFrame(rows)

    if strict and not report_df["keys_match_exactly"].all():
        bad = report_df.loc[~report_df["keys_match_exactly"]].copy()
        raise ValueError(
            "Los modelos no comparten exactamente las mismas llaves de evaluación. "
            "Esto rompe la comparabilidad justa.\n"
            f"{bad.to_string(index=False)}"
        )

    return report_df
    
def check_duplicate_keys(df_in: pd.DataFrame, name: str):
    dup_mask = df_in.duplicated(subset=[GROUP_COL, TIME_COL], keep=False)
    n_dup_rows = int(dup_mask.sum())
    n_dup_keys = int(df_in.loc[dup_mask, [GROUP_COL, TIME_COL]].drop_duplicates().shape[0])
    if n_dup_rows > 0:
        sample = df_in.loc[dup_mask, [GROUP_COL, TIME_COL, YEAR_COL, TARGET_COL]].head(20)
        raise ValueError(
            f"[{name}] Hay {n_dup_rows:,} filas duplicadas por (municipio, time_idx), "
            f"correspondientes a {n_dup_keys:,} llaves duplicadas.\n"
            f"Muestra:\n{sample.to_string(index=False)}"
        )

def check_timeidx_year_mapping(df_in: pd.DataFrame, name: str):
    tmp = df_in.groupby(TIME_COL)[YEAR_COL].nunique()
    bad = tmp[tmp > 1]
    if len(bad) > 0:
        raise ValueError(
            f"[{name}] time_idx no mapea de forma única a YEAR_COL. "
            f"Ejemplos:\n{bad.head(20).to_string()}"
        )

def continuity_report(df_in: pd.DataFrame) -> Dict[str, float]:
    ordered = df_in.sort_values([GROUP_COL, TIME_COL]).copy()
    diffs = ordered.groupby(GROUP_COL)[TIME_COL].diff().dropna()
    if len(diffs) == 0:
        return {
            "all_consecutive": True,
            "n_gaps": 0,
            "gap_rate": 0.0,
            "min_diff": np.nan,
            "max_diff": np.nan,
        }
    n_gaps = int((diffs != 1).sum())
    return {
        "all_consecutive": bool((diffs == 1).all()),
        "n_gaps": n_gaps,
        "gap_rate": float((diffs != 1).mean()),
        "min_diff": safe_float(diffs.min()),
        "max_diff": safe_float(diffs.max()),
    }

def dataset_report(df_in: pd.DataFrame, name: str) -> Dict[str, Any]:
    rep = {
        "name": name,
        "n_rows": int(len(df_in)),
        "n_municipalities": int(df_in[GROUP_COL].nunique()),
        "year_min": int(df_in[YEAR_COL].min()),
        "year_max": int(df_in[YEAR_COL].max()),
        "time_idx_min": int(df_in[TIME_COL].min()),
        "time_idx_max": int(df_in[TIME_COL].max()),
        "zero_rate": float((df_in[TARGET_COL] == 0).mean()),
        "target_mean": safe_float(df_in[TARGET_COL].mean()),
        "target_std": safe_float(df_in[TARGET_COL].std()),
        "target_max": safe_float(df_in[TARGET_COL].max()),
    }
    rep.update(continuity_report(df_in))
    return rep

def validate_dataframe(df_in: pd.DataFrame, name: str, save_path: Optional[Path] = None):
    x = ensure_string_group(df_in)
    check_required_non_nulls(x, name)
    check_duplicate_keys(x, name)
    check_timeidx_year_mapping(x, name)
    rep = dataset_report(x, name)
    print(f"\n[{name}] REPORTE")
    print(json.dumps(rep, indent=2))
    if save_path is not None:
        write_json(save_path, rep)
    return rep

def validate_split_logic(train_df: pd.DataFrame, eval_df: pd.DataFrame, eval_start_year: int, name: str):
    train_max_year = int(train_df[YEAR_COL].max())
    eval_min_year = int(eval_df[YEAR_COL].min())
    eval_max_year = int(eval_df[YEAR_COL].max())
    if train_max_year >= eval_start_year:
        raise ValueError(
            f"[{name}] Leakage temporal: train_max_year={train_max_year} debe ser < eval_start_year={eval_start_year}"
        )
    if eval_max_year < eval_start_year:
        raise ValueError(f"[{name}] eval_df no contiene el año de evaluación {eval_start_year}")
    print(
        f"[{name}] Split OK | train_max_year={train_max_year} | "
        f"eval_year_range=[{eval_min_year}, {eval_max_year}] | eval_start_year={eval_start_year}"
    )

# =========================
# 5. WEEKLY FEATURES
# =========================
def add_weekly_features(df_in: pd.DataFrame) -> pd.DataFrame:
    data = ensure_string_group(df_in.copy())
    data = data.sort_values([GROUP_COL, TIME_COL]).reset_index(drop=True)

    for c in CLIMATE_COLS + AGE_COLS + [FLOW_COL, TARGET_COL]:
        if c not in data.columns:
            data[c] = 0.0

    time_map = (
        data[[YEAR_COL, TIME_COL]]
        .drop_duplicates()
        .sort_values([YEAR_COL, TIME_COL])
        .copy()
    )
    time_map["week_in_year"] = time_map.groupby(YEAR_COL).cumcount() + 1
    data = data.merge(time_map, on=[YEAR_COL, TIME_COL], how="left")

    data["week_sin"] = np.sin(2 * np.pi * data["week_in_year"] / 52.0)
    data["week_cos"] = np.cos(2 * np.pi * data["week_in_year"] / 52.0)

    for lag in TARGET_LAGS:
        data[f"{TARGET_COL}_lag{lag}"] = data.groupby(GROUP_COL)[TARGET_COL].shift(lag)
    for w in TARGET_ROLLS:
        data[f"{TARGET_COL}_roll{w}"] = (
            data.groupby(GROUP_COL)[TARGET_COL]
            .transform(lambda s: s.shift(1).rolling(w, min_periods=1).mean())
        )

    for lag in FLOW_LAGS:
        data[f"{FLOW_COL}_lag{lag}"] = data.groupby(GROUP_COL)[FLOW_COL].shift(lag)
    for w in FLOW_ROLLS:
        data[f"{FLOW_COL}_roll{w}"] = (
            data.groupby(GROUP_COL)[FLOW_COL]
            .transform(lambda s: s.shift(1).rolling(w, min_periods=1).mean())
        )

    for col in AGE_COLS:
        for lag in AGE_LAGS:
            data[f"{col}_lag{lag}"] = data.groupby(GROUP_COL)[col].shift(lag)
        for w in AGE_ROLLS:
            data[f"{col}_roll{w}"] = (
                data.groupby(GROUP_COL)[col]
                .transform(lambda s: s.shift(1).rolling(w, min_periods=1).mean())
            )

    derived_cols = [c for c in data.columns if ("_lag" in c) or ("_roll" in c)] + [
        "week_in_year", "week_sin", "week_cos"
    ]
    for c in derived_cols:
        data[c] = data[c].fillna(0.0)

    base_fill_cols = CLIMATE_COLS + [FLOW_COL] + AGE_COLS + [TARGET_COL]
    for c in base_fill_cols:
        data[c] = data[c].fillna(0.0)

    return data

# =========================
# 6. OPTIONAL MUNICIPALITY FILTER
# =========================
def select_active_municipalities(train_df: pd.DataFrame) -> set:
    stats = (
        train_df.groupby(GROUP_COL)[TARGET_COL]
        .agg(
            total_cases="sum",
            positive_rate=lambda s: float((s > 0).mean()),
            max_cases="max",
        )
        .reset_index()
    )
    keep = stats[
        (stats["total_cases"] >= ACTIVE_MIN_TOTAL_CASES) |
        (stats["positive_rate"] >= ACTIVE_MIN_POSITIVE_RATE) |
        (stats["max_cases"] >= ACTIVE_MIN_MAX_CASES)
    ][GROUP_COL].astype(str).unique()
    return set(keep)

# =========================
# 7. DYNAMIC OUTBREAK REFERENCES (PAST-ONLY, BY MUNICIPALITY AND WEEK)
# =========================
def add_dynamic_outbreak_columns(df_in: pd.DataFrame) -> pd.DataFrame:
    """
    Adds a dynamic outbreak baseline and threshold for each municipality-week using only past data.

    Definition:
        baseline_t = mean of previous OUTBREAK_BASELINE_WINDOW observed weeks
                     (fallback to expanding past mean if early history is short)
        outbreak_threshold_t = max(
            OUTBREAK_MIN_CASES_ABS,
            (1.0 + OUTBREAK_GROWTH_RATE) * max(baseline_t, OUTBREAK_BASELINE_FLOOR)
        )

    Important:
    - uses shift(1), so the current week never leaks into its own threshold
    - works for both train and eval rows
    """
    out = df_in.copy()
    out = out.sort_values([GROUP_COL, TIME_COL]).reset_index(drop=True)

    def _per_muni(g: pd.DataFrame) -> pd.DataFrame:
        g = g.sort_values(TIME_COL).copy()

        y = g[TARGET_COL].astype(float)

        # past-only series
        y_lag = y.shift(1)

        # primary recent baseline: rolling mean of last 4 weeks
        baseline_roll = y_lag.rolling(
            window=OUTBREAK_BASELINE_WINDOW,
            min_periods=OUTBREAK_BASELINE_MIN_HISTORY
        ).mean()

        # fallback baseline for very early rows
        baseline_expand = y_lag.expanding(min_periods=1).mean()

        baseline = baseline_roll.fillna(baseline_expand).fillna(0.0).astype(float)

        threshold = np.maximum(
            OUTBREAK_MIN_CASES_ABS,
            (1.0 + OUTBREAK_GROWTH_RATE) * np.maximum(baseline.values, OUTBREAK_BASELINE_FLOOR)
        ).astype(float)

        g["outbreak_baseline_ref"] = baseline.values
        g["outbreak_case_threshold_ref"] = threshold
        g["is_outbreak_observed_dynamic"] = (g[TARGET_COL].astype(float).values >= threshold).astype(int)
        return g

    out = (
        out.groupby(GROUP_COL, group_keys=False)
        .apply(_per_muni)
        .reset_index(drop=True)
    )

    return out

# =========================
# 8. STATIC FEATURES + MUNICIPALITY/TIME-SPECIFIC WEIGHTS
# =========================
def add_fold_static_features_and_weights(
    train_df: pd.DataFrame,
    eval_df: pd.DataFrame
) -> Tuple[pd.DataFrame, pd.DataFrame, Dict[str, Any]]:
    train_df = train_df.copy()
    eval_df = eval_df.copy()

    train_for_stats = add_dynamic_outbreak_columns(train_df)

    stats = (
        train_for_stats.groupby(GROUP_COL)
        .agg(
            mun_std_cases_hist=(TARGET_COL, "std"),
            mun_positive_rate_hist=(TARGET_COL, lambda s: float((s > 0).mean())),
        )
        .reset_index()
    )

    global_fill = {
        "mun_std_cases_hist": float(train_df[TARGET_COL].std()) if len(train_df) > 1 else 0.0,
        "mun_positive_rate_hist": float((train_df[TARGET_COL] > 0).mean()),
    }

    def _merge_static(d: pd.DataFrame) -> pd.DataFrame:
        out = d.merge(stats, on=GROUP_COL, how="left")
        for c, v in global_fill.items():
            out[c] = out[c].fillna(v)
        return out

    train_aug = _merge_static(train_df)
    eval_aug = _merge_static(eval_df)

    # Add dynamic outbreak references using only past data
    train_aug = add_dynamic_outbreak_columns(train_aug)
    eval_aug = add_dynamic_outbreak_columns(eval_aug)

    def _assign_weights(d: pd.DataFrame) -> pd.DataFrame:
        out = d.copy()

        y = out[TARGET_COL].astype(float).values
        thr = out["outbreak_case_threshold_ref"].astype(float).values

        w = np.select(
            [
                y == 0,
                (y > 0) & (y < thr),
                y >= thr,
            ],
            [
                LOSS_W_ZERO,
                LOSS_W_POSITIVE_NON_OUTBREAK,
                LOSS_W_OUTBREAK,
            ],
            default=LOSS_W_ZERO,
        ).astype(float)

        regime = np.select(
            [
                y == 0,
                (y > 0) & (y < thr),
                y >= thr,
            ],
            [
                "zero",
                "positive_non_outbreak",
                "outbreak",
            ],
            default="zero",
        )

        out["loss_weight"] = w
        out["loss_regime"] = regime
        return out

    train_aug = _assign_weights(train_aug)
    eval_aug = _assign_weights(eval_aug)

    weight_info = {
        "weight_scheme": "dynamic_outbreak_single_class",
        "outbreak_definition": (
            f"y_t >= max({OUTBREAK_MIN_CASES_ABS}, "
            f"(1+{OUTBREAK_GROWTH_RATE}) * baseline_t), "
            f"baseline_t = mean(previous {OUTBREAK_BASELINE_WINDOW} weeks, past-only)"
        ),
        "OUTBREAK_BASELINE_WINDOW": int(OUTBREAK_BASELINE_WINDOW),
        "OUTBREAK_BASELINE_MIN_HISTORY": int(OUTBREAK_BASELINE_MIN_HISTORY),
        "OUTBREAK_GROWTH_RATE": float(OUTBREAK_GROWTH_RATE),
        "OUTBREAK_MIN_CASES_ABS": float(OUTBREAK_MIN_CASES_ABS),
        "LOSS_W_ZERO": float(LOSS_W_ZERO),
        "LOSS_W_POSITIVE_NON_OUTBREAK": float(LOSS_W_POSITIVE_NON_OUTBREAK),
        "LOSS_W_OUTBREAK": float(LOSS_W_OUTBREAK),
        "train_loss_regime_counts": train_aug["loss_regime"].value_counts(dropna=False).to_dict(),
        "eval_loss_regime_counts": eval_aug["loss_regime"].value_counts(dropna=False).to_dict(),
    }

    return train_aug, eval_aug, weight_info

# =========================
# 9. METRICS
# =========================
def smape(y_true, y_pred, eps=1e-8):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return 100.0 * np.mean(
        2.0 * np.abs(y_pred - y_true) / (np.abs(y_true) + np.abs(y_pred) + eps)
    )

def binary_metrics(y_true_bin, y_pred_bin):
    y_true_bin = np.asarray(y_true_bin, dtype=int)
    y_pred_bin = np.asarray(y_pred_bin, dtype=int)

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true_bin, y_pred_bin, average="binary", zero_division=0
    )

    tp = int(np.sum((y_true_bin == 1) & (y_pred_bin == 1)))
    tn = int(np.sum((y_true_bin == 0) & (y_pred_bin == 0)))
    fp = int(np.sum((y_true_bin == 0) & (y_pred_bin == 1)))
    fn = int(np.sum((y_true_bin == 1) & (y_pred_bin == 0)))

    false_alarm_rate = float(fp / (fp + tn)) if (fp + tn) > 0 else np.nan
    missed_rate = float(fn / (fn + tp)) if (fn + tp) > 0 else np.nan
    pred_positive_rate = float(np.mean(y_pred_bin == 1))
    true_positive_rate = float(np.mean(y_true_bin == 1))

    specificity = float(tn / (tn + fp)) if (tn + fp) > 0 else np.nan
    balanced_accuracy = float((recall + specificity) / 2.0) if not pd.isna(specificity) else np.nan

    return {
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
        "false_alarm_rate": false_alarm_rate,
        "missed_rate": missed_rate,
        "pred_positive_rate": pred_positive_rate,
        "true_positive_rate": true_positive_rate,
        "specificity": specificity,
        "balanced_accuracy": balanced_accuracy,
        "tp": tp,
        "tn": tn,
        "fp": fp,
        "fn": fn,
    }

def compute_regression_metrics(y_true, y_pred, positive_threshold=POSITIVE_CLASSIFICATION_THRESHOLD):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    out = {}
    out["n_obs"] = int(len(y_true))
    out["true_zero_rate"] = float(np.mean(y_true == 0))
    out["pred_exact_zero_rate"] = float(np.mean(y_pred == 0))
    out["pred_below_positive_threshold_rate"] = float(np.mean(y_pred < positive_threshold))
    out["MAE"] = float(mean_absolute_error(y_true, y_pred))
    out["RMSE"] = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    out["sMAPE"] = float(smape(y_true, y_pred))
    try:
        out["R2"] = float(r2_score(y_true, y_pred))
    except Exception:
        out["R2"] = np.nan

    mask_pos = y_true > 0
    if mask_pos.sum() > 0:
        out["MAE_nonzero"] = float(mean_absolute_error(y_true[mask_pos], y_pred[mask_pos]))
        out["RMSE_nonzero"] = float(np.sqrt(mean_squared_error(y_true[mask_pos], y_pred[mask_pos])))
        out["sMAPE_nonzero"] = float(smape(y_true[mask_pos], y_pred[mask_pos]))
        out["mean_true_when_positive"] = float(np.mean(y_true[mask_pos]))
        out["mean_pred_when_positive"] = float(np.mean(y_pred[mask_pos]))
        out["median_true_when_positive"] = float(np.median(y_true[mask_pos]))
        out["median_pred_when_positive"] = float(np.median(y_pred[mask_pos]))
    else:
        out["MAE_nonzero"] = np.nan
        out["RMSE_nonzero"] = np.nan
        out["sMAPE_nonzero"] = np.nan
        out["mean_true_when_positive"] = np.nan
        out["mean_pred_when_positive"] = np.nan
        out["median_true_when_positive"] = np.nan
        out["median_pred_when_positive"] = np.nan

    mask_zero = y_true == 0
    if mask_zero.sum() > 0:
        out["mean_pred_when_true_zero"] = float(np.mean(y_pred[mask_zero]))
        out["median_pred_when_true_zero"] = float(np.median(y_pred[mask_zero]))
    else:
        out["mean_pred_when_true_zero"] = np.nan
        out["median_pred_when_true_zero"] = np.nan

    return out

def add_common_label_columns(
    pred_df: pd.DataFrame,
    positive_threshold: float = POSITIVE_CLASSIFICATION_THRESHOLD,
) -> pd.DataFrame:
    out = pred_df.copy()
    out["positive_threshold"] = float(positive_threshold)
    out["outbreak_relaxed_factor"] = float(OUTBREAK_RELAXED_FACTOR)

    if "outbreak_case_threshold_ref" not in out.columns:
        raise ValueError("Falta 'outbreak_case_threshold_ref' en pred_df antes de etiquetar outbreaks.")

    # No recalcula un threshold nuevo desde cero:
    # solo usa el threshold real ya existente y le aplica el factor relaxed.
    out["outbreak_case_threshold_ref_relaxed"] = (
        out["outbreak_case_threshold_ref"].astype(float) * float(OUTBREAK_RELAXED_FACTOR)
    )

    # Positive labels
    out["is_positive_true"] = (out["y_true"] > 0).astype(int)
    out["is_positive_pred_model"] = (out["y_pred"] >= float(positive_threshold)).astype(int)
    out["is_positive_pred_naive"] = (out["naive_pred_lag1"] >= float(positive_threshold)).astype(int)

    # Strict outbreak truth (ground truth unchanged)
    out["is_outbreak_true"] = (out["y_true"] >= out["outbreak_case_threshold_ref"]).astype(int)

    # Strict outbreak prediction
    out["is_outbreak_pred_model"] = (out["y_pred"] >= out["outbreak_case_threshold_ref"]).astype(int)
    out["is_outbreak_pred_naive"] = (out["naive_pred_lag1"] >= out["outbreak_case_threshold_ref"]).astype(int)

    # Relaxed prediction ONLY
    out["is_outbreak_pred_model_relaxed"] = (
        out["y_pred"] >= out["outbreak_case_threshold_ref_relaxed"]
    ).astype(int)
    out["is_outbreak_pred_naive_relaxed"] = (
        out["naive_pred_lag1"] >= out["outbreak_case_threshold_ref_relaxed"]
    ).astype(int)

    out["outbreak_threshold_rule"] = (
        f"strict truth: y_t >= max({OUTBREAK_MIN_CASES_ABS}, "
        f"(1+{OUTBREAK_GROWTH_RATE})*baseline_t), "
        f"baseline_t=mean(previous {OUTBREAK_BASELINE_WINDOW} weeks); "
        f"relaxed prediction eval uses y_pred >= {OUTBREAK_RELAXED_FACTOR:.2f} * strict_threshold"
    )
    return out

def compute_overall_and_horizon_metrics(
    pred_df: pd.DataFrame,
    model_name: str,
    fold_name: str,
    partition_name: str,
    positive_threshold: float = POSITIVE_CLASSIFICATION_THRESHOLD,
) -> Tuple[Dict[str, Any], pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame, Optional[pd.DataFrame]]:
    df_eval = pred_df.copy()

    overall = compute_regression_metrics(
        y_true=df_eval["y_true"].values,
        y_pred=df_eval["y_pred"].values,
        positive_threshold=positive_threshold,
    )

    overall["model_name"] = model_name
    overall["fold_name"] = fold_name
    overall["partition_name"] = partition_name
    overall["positive_threshold"] = float(positive_threshold)
    overall["outbreak_relaxed_factor"] = float(OUTBREAK_RELAXED_FACTOR)
    overall["outbreak_threshold_rule"] = (
        f"strict truth: y_t >= max({OUTBREAK_MIN_CASES_ABS}, "
        f"(1+{OUTBREAK_GROWTH_RATE})*baseline_t), "
        f"baseline_t=mean(previous {OUTBREAK_BASELINE_WINDOW} weeks); "
        f"relaxed prediction eval uses y_pred >= {OUTBREAK_RELAXED_FACTOR:.2f} * strict_threshold"
    )

    try:
        overall["pearson_corr"] = float(pd.Series(df_eval["y_true"]).corr(pd.Series(df_eval["y_pred"]), method="pearson"))
    except Exception:
        overall["pearson_corr"] = np.nan
    try:
        overall["spearman_corr"] = float(pd.Series(df_eval["y_true"]).corr(pd.Series(df_eval["y_pred"]), method="spearman"))
    except Exception:
        overall["spearman_corr"] = np.nan

    # Positive classification
    pos_model = binary_metrics(df_eval["is_positive_true"].values, df_eval["is_positive_pred_model"].values)

    overall["positive_model_precision"] = pos_model["precision"]
    overall["positive_model_recall"] = pos_model["recall"]
    overall["positive_model_f1"] = pos_model["f1"]
    overall["positive_model_false_alarm_rate"] = pos_model["false_alarm_rate"]
    overall["positive_model_specificity"] = pos_model["specificity"]
    overall["positive_model_balanced_accuracy"] = pos_model["balanced_accuracy"]

    # Strict outbreak evaluation: strict truth vs strict prediction
    out_model = binary_metrics(
        df_eval["is_outbreak_true"].values,
        df_eval["is_outbreak_pred_model"].values
    )

    overall["outbreak_model_precision"] = out_model["precision"]
    overall["outbreak_model_recall"] = out_model["recall"]
    overall["outbreak_model_f1"] = out_model["f1"]
    overall["outbreak_model_missed_rate"] = out_model["missed_rate"]
    overall["outbreak_model_false_alarm_rate"] = out_model["false_alarm_rate"]
    overall["outbreak_model_specificity"] = out_model["specificity"]
    overall["outbreak_model_balanced_accuracy"] = out_model["balanced_accuracy"]
    overall["outbreak_model_positive_rate"] = out_model["pred_positive_rate"]

    # Relaxed outbreak evaluation: strict truth vs relaxed prediction
    out_model_relaxed = binary_metrics(
        df_eval["is_outbreak_true"].values,
        df_eval["is_outbreak_pred_model_relaxed"].values
    )

    overall["outbreak_model_precision_relaxed"] = out_model_relaxed["precision"]
    overall["outbreak_model_recall_relaxed"] = out_model_relaxed["recall"]
    overall["outbreak_model_f1_relaxed"] = out_model_relaxed["f1"]
    overall["outbreak_model_missed_rate_relaxed"] = out_model_relaxed["missed_rate"]
    overall["outbreak_model_false_alarm_rate_relaxed"] = out_model_relaxed["false_alarm_rate"]
    overall["outbreak_model_specificity_relaxed"] = out_model_relaxed["specificity"]
    overall["outbreak_model_balanced_accuracy_relaxed"] = out_model_relaxed["balanced_accuracy"]
    overall["outbreak_model_positive_rate_relaxed"] = out_model_relaxed["pred_positive_rate"]

    # Regression restricted to STRICT true outbreaks only
    outbreak_mask = df_eval["is_outbreak_true"] == 1
    if outbreak_mask.sum() > 0:
        y_out = df_eval.loc[outbreak_mask, "y_true"].values
        yhat_out = df_eval.loc[outbreak_mask, "y_pred"].values

        overall["MAE_on_true_outbreaks"] = float(mean_absolute_error(y_out, yhat_out))
        overall["RMSE_on_true_outbreaks"] = float(np.sqrt(mean_squared_error(y_out, yhat_out)))
        overall["sMAPE_on_true_outbreaks"] = float(smape(y_out, yhat_out))

        err_model = yhat_out - y_out
        mean_true_out = float(np.mean(y_out)) if len(y_out) > 0 else np.nan

        overall["mean_signed_error_on_outbreaks"] = float(np.mean(err_model))
        overall["mean_abs_error_on_outbreaks"] = float(np.mean(np.abs(err_model)))
        overall["median_signed_error_on_outbreaks"] = float(np.median(err_model))

        overall["mean_true_on_outbreaks"] = float(np.mean(y_out))
        overall["mean_pred_on_outbreaks"] = float(np.mean(yhat_out))

        overall["relative_bias_on_outbreaks"] = float(np.mean(err_model) / mean_true_out) if mean_true_out != 0 else np.nan
        overall["underprediction_rate_on_outbreaks"] = float(np.mean(yhat_out < y_out))
        overall["overprediction_rate_on_outbreaks"] = float(np.mean(yhat_out > y_out))
    else:
        for c in [
            "MAE_on_true_outbreaks", "RMSE_on_true_outbreaks", "sMAPE_on_true_outbreaks",
            "mean_signed_error_on_outbreaks", "mean_abs_error_on_outbreaks", "median_signed_error_on_outbreaks",
            "mean_true_on_outbreaks", "mean_pred_on_outbreaks",
            "relative_bias_on_outbreaks",
            "underprediction_rate_on_outbreaks", "overprediction_rate_on_outbreaks",
        ]:
            overall[c] = np.nan

    regression_rows = []
    positive_rows = []
    outbreak_rows = []
    outbreak_confusion_rows = []

    for h in sorted(df_eval["horizon"].unique()):
        g = df_eval.loc[df_eval["horizon"] == h].copy()

        met = compute_regression_metrics(
            y_true=g["y_true"].values,
            y_pred=g["y_pred"].values,
            positive_threshold=positive_threshold,
        )

        reg_row = dict(met)
        reg_row["model_name"] = model_name
        reg_row["fold_name"] = fold_name
        reg_row["partition_name"] = partition_name
        reg_row["horizon"] = int(h)

        try:
            reg_row["pearson_corr"] = float(pd.Series(g["y_true"]).corr(pd.Series(g["y_pred"]), method="pearson"))
        except Exception:
            reg_row["pearson_corr"] = np.nan
        try:
            reg_row["spearman_corr"] = float(pd.Series(g["y_true"]).corr(pd.Series(g["y_pred"]), method="spearman"))
        except Exception:
            reg_row["spearman_corr"] = np.nan

        regression_rows.append(reg_row)

        pos_model_h = binary_metrics(g["is_positive_true"].values, g["is_positive_pred_model"].values)
        positive_rows.append({
            "model_name": model_name,
            "fold_name": fold_name,
            "partition_name": partition_name,
            "horizon": int(h),
            "n_obs": int(len(g)),
            "n_true_positive_obs": int(g["is_positive_true"].sum()),
            "true_positive_rate": float(g["is_positive_true"].mean()),
            "model_precision": pos_model_h["precision"],
            "model_recall": pos_model_h["recall"],
            "model_f1": pos_model_h["f1"],
            "model_false_alarm_rate": pos_model_h["false_alarm_rate"],
            "model_specificity": pos_model_h["specificity"],
            "model_balanced_accuracy": pos_model_h["balanced_accuracy"],
            "model_tp": pos_model_h["tp"],
            "model_fp": pos_model_h["fp"],
            "model_tn": pos_model_h["tn"],
            "model_fn": pos_model_h["fn"],
        })

        out_model_h = binary_metrics(
            g["is_outbreak_true"].values,
            g["is_outbreak_pred_model"].values
        )

        out_model_h_relaxed = binary_metrics(
            g["is_outbreak_true"].values,
            g["is_outbreak_pred_model_relaxed"].values
        )

        outbreak_mask_h = g["is_outbreak_true"] == 1
        if outbreak_mask_h.sum() > 0:
            y_out_h = g.loc[outbreak_mask_h, "y_true"].values
            yhat_out_h = g.loc[outbreak_mask_h, "y_pred"].values

            mae_out_h = float(mean_absolute_error(y_out_h, yhat_out_h))
            rmse_out_h = float(np.sqrt(mean_squared_error(y_out_h, yhat_out_h)))
            smape_out_h = float(smape(y_out_h, yhat_out_h))

            mean_true_out_h = float(np.mean(y_out_h))
            rel_bias_h = float(np.mean(yhat_out_h - y_out_h) / mean_true_out_h) if mean_true_out_h != 0 else np.nan

            under_h = float(np.mean(yhat_out_h < y_out_h))
            over_h = float(np.mean(yhat_out_h > y_out_h))
        else:
            mae_out_h = np.nan
            rmse_out_h = np.nan
            smape_out_h = np.nan
            rel_bias_h = np.nan
            under_h = np.nan
            over_h = np.nan

        outbreak_rows.append({
            "model_name": model_name,
            "fold_name": fold_name,
            "partition_name": partition_name,
            "horizon": int(h),
            "n_obs": int(len(g)),

            # Strict
            "n_true_outbreaks": int(g["is_outbreak_true"].sum()),
            "true_outbreak_rate": float(g["is_outbreak_true"].mean()),
            "model_detected_outbreak_obs": int(out_model_h["tp"]),
            "model_missed_outbreak_obs": int(out_model_h["fn"]),
            "model_false_alarm_outbreak_obs": int(out_model_h["fp"]),
            "model_precision": out_model_h["precision"],
            "model_recall": out_model_h["recall"],
            "model_f1": out_model_h["f1"],
            "model_false_alarm_rate": out_model_h["false_alarm_rate"],
            "model_specificity": out_model_h["specificity"],
            "model_balanced_accuracy": out_model_h["balanced_accuracy"],

            # Relaxed prediction vs strict truth
            "model_detected_outbreak_obs_relaxed": int(out_model_h_relaxed["tp"]),
            "model_missed_outbreak_obs_relaxed": int(out_model_h_relaxed["fn"]),
            "model_false_alarm_outbreak_obs_relaxed": int(out_model_h_relaxed["fp"]),
            "model_precision_relaxed": out_model_h_relaxed["precision"],
            "model_recall_relaxed": out_model_h_relaxed["recall"],
            "model_f1_relaxed": out_model_h_relaxed["f1"],
            "model_false_alarm_rate_relaxed": out_model_h_relaxed["false_alarm_rate"],
            "model_specificity_relaxed": out_model_h_relaxed["specificity"],
            "model_balanced_accuracy_relaxed": out_model_h_relaxed["balanced_accuracy"],

            # Errors on strict outbreaks only
            "MAE_on_true_outbreaks": mae_out_h,
            "RMSE_on_true_outbreaks": rmse_out_h,
            "sMAPE_on_true_outbreaks": smape_out_h,
            "relative_bias_on_outbreaks": rel_bias_h,
            "underprediction_rate_on_outbreaks": under_h,
            "overprediction_rate_on_outbreaks": over_h,
        })

        outbreak_confusion_rows.append({
            "model_name": model_name,
            "fold_name": fold_name,
            "partition_name": partition_name,
            "horizon": int(h),
            "strict_tp": int(out_model_h["tp"]),
            "strict_fp": int(out_model_h["fp"]),
            "strict_tn": int(out_model_h["tn"]),
            "strict_fn": int(out_model_h["fn"]),
            "relaxed_tp": int(out_model_h_relaxed["tp"]),
            "relaxed_fp": int(out_model_h_relaxed["fp"]),
            "relaxed_tn": int(out_model_h_relaxed["tn"]),
            "relaxed_fn": int(out_model_h_relaxed["fn"]),
        })

    regression_by_horizon = pd.DataFrame(regression_rows)
    positive_by_horizon = pd.DataFrame(positive_rows)
    outbreak_by_horizon = pd.DataFrame(outbreak_rows)
    outbreak_confusion_by_horizon = pd.DataFrame(outbreak_confusion_rows)

    metrics_by_municipality = None
    if SAVE_METRICS_BY_MUNICIPALITY:
        rows = []
        for muni, g in df_eval.groupby(GROUP_COL):
            met = compute_regression_metrics(
                y_true=g["y_true"].values,
                y_pred=g["y_pred"].values,
                positive_threshold=positive_threshold,
            )
            pos_m = binary_metrics(g["is_positive_true"].values, g["is_positive_pred_model"].values)
            out_m = binary_metrics(g["is_outbreak_true"].values, g["is_outbreak_pred_model"].values)
            out_m_relaxed = binary_metrics(
                g["is_outbreak_true"].values,
                g["is_outbreak_pred_model_relaxed"].values
            )

            row = {
                GROUP_COL: muni,
                "model_name": model_name,
                "fold_name": fold_name,
                "partition_name": partition_name,
                **met,
                "positive_model_f1": pos_m["f1"],
                "positive_model_recall": pos_m["recall"],
                "positive_model_precision": pos_m["precision"],
                "outbreak_model_f1": out_m["f1"],
                "outbreak_model_recall": out_m["recall"],
                "outbreak_model_precision": out_m["precision"],
                "outbreak_model_f1_relaxed": out_m_relaxed["f1"],
                "outbreak_model_recall_relaxed": out_m_relaxed["recall"],
                "outbreak_model_precision_relaxed": out_m_relaxed["precision"],
            }
            rows.append(row)
        metrics_by_municipality = pd.DataFrame(rows)

    return (
        overall,
        regression_by_horizon,
        positive_by_horizon,
        outbreak_by_horizon,
        outbreak_confusion_by_horizon,
        metrics_by_municipality,
    )

# =========================
# 10. PREDICTION FRAME UTILITIES
# =========================
def add_naive_lag1(pred_df: pd.DataFrame, source_df: pd.DataFrame) -> pd.DataFrame:
    source = (
        source_df[[GROUP_COL, TIME_COL, TARGET_COL]]
        .copy()
        .drop_duplicates([GROUP_COL, TIME_COL])
        .rename(columns={TIME_COL: "naive_time_idx", TARGET_COL: "naive_pred_lag1"})
    )
    source[GROUP_COL] = source[GROUP_COL].astype(str)

    out = pred_df.copy()
    out["naive_time_idx"] = out["prediction_start_time_idx"] - 1
    out = out.merge(source, on=[GROUP_COL, "naive_time_idx"], how="left")
    out["naive_pred_lag1"] = out["naive_pred_lag1"].fillna(0.0).astype(float)
    return out

def collapse_duplicate_predictions(pred_df: pd.DataFrame) -> pd.DataFrame:
    group_cols = [GROUP_COL, "prediction_start_time_idx", "horizon", "target_time_idx"]

    out = pred_df.copy()
    out[GROUP_COL] = out[GROUP_COL].astype(str)
    out["prediction_start_time_idx"] = out["prediction_start_time_idx"].astype(int)
    out["horizon"] = out["horizon"].astype(int)
    out["target_time_idx"] = out["target_time_idx"].astype(int)
    out["y_true"] = out["y_true"].astype(float)
    out["y_pred"] = out["y_pred"].astype(float)

    agg_df = (
        out.groupby(group_cols, as_index=False)
        .agg(
            y_true=("y_true", "first"),
            y_pred=("y_pred", "mean"),
        )
    )
    return agg_df

def prepare_predictions_for_evaluation(
    pred_df: pd.DataFrame,
    source_eval_df: pd.DataFrame,
) -> pd.DataFrame:
    out = pred_df.copy()
    out = collapse_duplicate_predictions(out)

    out[GROUP_COL] = out[GROUP_COL].astype(str)
    out["prediction_start_time_idx"] = out["prediction_start_time_idx"].astype(int)
    out["horizon"] = out["horizon"].astype(int)
    out["target_time_idx"] = out["target_time_idx"].astype(int)
    out["y_pred"] = out["y_pred"].clip(lower=0.0).astype(float)
    out["y_true"] = out["y_true"].astype(float)

    out = add_naive_lag1(out, source_eval_df)

    outbreak_ref = (
        source_eval_df[
            [GROUP_COL, TIME_COL, "outbreak_baseline_ref", "outbreak_case_threshold_ref"]
        ]
        .drop_duplicates([GROUP_COL, TIME_COL])
        .rename(columns={TIME_COL: "target_time_idx"})
        .copy()
    )
    outbreak_ref[GROUP_COL] = outbreak_ref[GROUP_COL].astype(str)
    outbreak_ref["target_time_idx"] = outbreak_ref["target_time_idx"].astype(int)

    out = out.merge(
        outbreak_ref,
        on=[GROUP_COL, "target_time_idx"],
        how="left",
        validate="many_to_one",
    )

    if out["outbreak_case_threshold_ref"].isnull().any():
        missing = out.loc[out["outbreak_case_threshold_ref"].isnull(), [GROUP_COL, "target_time_idx"]].head(20)
        raise ValueError(
            "Faltan outbreak thresholds dinámicos al preparar predicciones. "
            f"Muestra:\n{missing.to_string(index=False)}"
        )

    out = add_common_label_columns(
        out,
        positive_threshold=POSITIVE_CLASSIFICATION_THRESHOLD,
    )
    return out

# =========================
# 11. EVENT / ANTICIPATION REPORTS
# =========================
def build_outbreak_anticipation_report(pred_df: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Reporte de anticipación de eventos de brote a nivel de evento.

    Convención temporal del pipeline:
        target_time_idx      = prediction_start_time_idx + (horizon - 1)
        info_cutoff_time_idx = prediction_start_time_idx - 1
                             = target_time_idx - horizon

    Definición de anticipación a nivel de evento:
        anticipation_weeks = max( event_start_time_idx - info_cutoff - 1, 0 )
        sobre todas las predicciones positivas dentro del evento.

    Para eventos de una sola semana esta fórmula colapsa a horizon - 1.
    Para eventos multi-semana, la fórmula incorpora correctamente el
    desplazamiento entre la semana objetivo y el inicio del evento, evitando
    sobreestimar la anticipación. Por ejemplo, un evento {10, 11, 12} cuya
    semana 12 se predice desde info_cutoff=9 (horizonte 3) tiene
    anticipación = max(10 - 9 - 1, 0) = 0 semanas, no 2.

    Este reporte incluye:
    - strict anticipation: strict truth vs strict prediction
    - relaxed anticipation: strict truth vs relaxed prediction
    """
    df_out = pred_df.loc[pred_df["is_outbreak_true"] == 1].copy()

    if len(df_out) == 0:
        empty_event = pd.DataFrame(columns=[
            GROUP_COL, "event_id", "event_start_time_idx", "event_end_time_idx",
            "event_duration_weeks", "peak_y_true",

            "model_detected", "model_max_detected_horizon", "model_max_anticipation_weeks",
            "model_detected_relaxed", "model_max_detected_horizon_relaxed", "model_max_anticipation_weeks_relaxed",

            "naive_detected", "naive_max_detected_horizon", "naive_max_anticipation_weeks",
            "naive_detected_relaxed", "naive_max_detected_horizon_relaxed", "naive_max_anticipation_weeks_relaxed",
        ])

        empty_summary = pd.DataFrame([{
            "n_true_outbreak_events": 0,

            "definition_max_anticipation_weeks": "max(event_start_time_idx - (target_time_idx - horizon) - 1) over predicted-outbreak rows within the event, clamped at 0",
            "definition_horizon_1": "horizon=1 means the prediction targets the next week using info up to the previous week",
            "definition_horizon_2": "horizon=2 means the prediction targets 2 weeks ahead",
            "definition_horizon_3": "horizon=3 means the prediction targets 3 weeks ahead",
            "definition_horizon_4": "horizon=4 means the prediction targets 4 weeks ahead",
            "anticipation_relative_to": "event_start_time_idx (NOT the specific outbreak week being predicted)",

            # strict model
            "model_detected_any": 0,
            "model_detected_any_rate": np.nan,
            "model_detected_with_0_week_anticipation_only": 0,
            "model_detected_with_1plus_week": 0,
            "model_detected_with_2plus_weeks": 0,
            "model_detected_with_3plus_weeks": 0,
            "model_mean_max_anticipation_weeks_on_detected": np.nan,
            "model_median_max_anticipation_weeks_on_detected": np.nan,
            "model_missed_all": 0,

            # relaxed model
            "model_detected_any_relaxed": 0,
            "model_detected_any_rate_relaxed": np.nan,
            "model_detected_with_0_week_anticipation_only_relaxed": 0,
            "model_detected_with_1plus_week_relaxed": 0,
            "model_detected_with_2plus_weeks_relaxed": 0,
            "model_detected_with_3plus_weeks_relaxed": 0,
            "model_mean_max_anticipation_weeks_on_detected_relaxed": np.nan,
            "model_median_max_anticipation_weeks_on_detected_relaxed": np.nan,
            "model_missed_all_relaxed": 0,

            # strict naive
            "naive_detected_any": 0,
            "naive_detected_any_rate": np.nan,
            "naive_detected_with_0_week_anticipation_only": 0,
            "naive_detected_with_1plus_week": 0,
            "naive_detected_with_2plus_weeks": 0,
            "naive_detected_with_3plus_weeks": 0,
            "naive_mean_max_anticipation_weeks_on_detected": np.nan,
            "naive_median_max_anticipation_weeks_on_detected": np.nan,
            "naive_missed_all": 0,

            # relaxed naive
            "naive_detected_any_relaxed": 0,
            "naive_detected_any_rate_relaxed": np.nan,
            "naive_detected_with_0_week_anticipation_only_relaxed": 0,
            "naive_detected_with_1plus_week_relaxed": 0,
            "naive_detected_with_2plus_weeks_relaxed": 0,
            "naive_detected_with_3plus_weeks_relaxed": 0,
            "naive_mean_max_anticipation_weeks_on_detected_relaxed": np.nan,
            "naive_median_max_anticipation_weeks_on_detected_relaxed": np.nan,
            "naive_missed_all_relaxed": 0,
        }])
        return empty_event, empty_summary

    weeks = (
        df_out[[GROUP_COL, "target_time_idx", "y_true"]]
        .drop_duplicates()
        .sort_values([GROUP_COL, "target_time_idx"])
        .copy()
    )

    weeks["gap"] = weeks.groupby(GROUP_COL)["target_time_idx"].diff()
    weeks["new_event"] = ((weeks["gap"].isna()) | (weeks["gap"] > 1)).astype(int)
    weeks["event_id"] = weeks.groupby(GROUP_COL)["new_event"].cumsum().astype(int)

    df_out = df_out.merge(
        weeks[[GROUP_COL, "target_time_idx", "event_id"]],
        on=[GROUP_COL, "target_time_idx"],
        how="left",
        validate="many_to_one",
    )

    def _lead_from_predictions(g: pd.DataFrame, mask_col: str, event_start: int) -> Tuple[int, int, int]:
        """
        Calcula (detected, max_horizon, max_lead) para un evento, midiendo
        anticipación correctamente contra el inicio del evento.
        """
        sub = g.loc[g[mask_col] == 1]
        if len(sub) == 0:
            return 0, 0, 0
        info_cutoff = sub["target_time_idx"].astype(int) - sub["horizon"].astype(int)
        leads = (event_start - info_cutoff - 1).clip(lower=0).astype(int)
        return 1, int(sub["horizon"].astype(int).max()), int(leads.max())

    def _event_level_summary(g: pd.DataFrame) -> pd.Series:
        event_start = int(g["target_time_idx"].min())
        event_end = int(g["target_time_idx"].max())

        # strict
        m_det, m_max_h, m_lead = _lead_from_predictions(g, "is_outbreak_pred_model", event_start)
        n_det, n_max_h, n_lead = _lead_from_predictions(g, "is_outbreak_pred_naive", event_start)

        # relaxed
        mr_det, mr_max_h, mr_lead = _lead_from_predictions(g, "is_outbreak_pred_model_relaxed", event_start)
        nr_det, nr_max_h, nr_lead = _lead_from_predictions(g, "is_outbreak_pred_naive_relaxed", event_start)

        return pd.Series({
            "event_start_time_idx": event_start,
            "event_end_time_idx": event_end,
            "event_duration_weeks": int(event_end - event_start + 1),
            "peak_y_true": float(g["y_true"].max()),

            # strict
            "model_detected": int(m_det),
            "model_max_detected_horizon": int(m_max_h),
            "model_max_anticipation_weeks": int(m_lead),

            "naive_detected": int(n_det),
            "naive_max_detected_horizon": int(n_max_h),
            "naive_max_anticipation_weeks": int(n_lead),

            # relaxed
            "model_detected_relaxed": int(mr_det),
            "model_max_detected_horizon_relaxed": int(mr_max_h),
            "model_max_anticipation_weeks_relaxed": int(mr_lead),

            "naive_detected_relaxed": int(nr_det),
            "naive_max_detected_horizon_relaxed": int(nr_max_h),
            "naive_max_anticipation_weeks_relaxed": int(nr_lead),
        })

    event_level = (
        df_out.groupby([GROUP_COL, "event_id"], as_index=False)
        .apply(_event_level_summary, include_groups=False)
        .reset_index(drop=True)
    )

    n_events = int(len(event_level))

    def _summary_for(prefix: str, detected_col: str, ant_col: str, suffix: str = "") -> dict:
        detected_mask = event_level[detected_col] == 1
        missed_mask = event_level[detected_col] == 0
        ant = event_level.loc[detected_mask, ant_col]

        return {
            f"{prefix}_detected_any{suffix}": int(detected_mask.sum()),
            f"{prefix}_detected_any_rate{suffix}": float(detected_mask.mean()) if n_events > 0 else np.nan,
            f"{prefix}_detected_with_0_week_anticipation_only{suffix}": int(((event_level[ant_col] == 0) & detected_mask).sum()),
            f"{prefix}_detected_with_1plus_week{suffix}": int(((event_level[ant_col] >= 1) & detected_mask).sum()),
            f"{prefix}_detected_with_2plus_weeks{suffix}": int(((event_level[ant_col] >= 2) & detected_mask).sum()),
            f"{prefix}_detected_with_3plus_weeks{suffix}": int(((event_level[ant_col] >= 3) & detected_mask).sum()),
            f"{prefix}_mean_max_anticipation_weeks_on_detected{suffix}": float(ant.mean()) if len(ant) > 0 else np.nan,
            f"{prefix}_median_max_anticipation_weeks_on_detected{suffix}": float(ant.median()) if len(ant) > 0 else np.nan,
            f"{prefix}_missed_all{suffix}": int(missed_mask.sum()),
        }

    summary = {
        "n_true_outbreak_events": n_events,
        "definition_max_anticipation_weeks": "max(event_start_time_idx - (target_time_idx - horizon) - 1) over predicted-outbreak rows within the event, clamped at 0",
        "definition_horizon_1": "horizon=1 means the prediction targets the next week using info up to the previous week",
        "definition_horizon_2": "horizon=2 means the prediction targets 2 weeks ahead",
        "definition_horizon_3": "horizon=3 means the prediction targets 3 weeks ahead",
        "definition_horizon_4": "horizon=4 means the prediction targets 4 weeks ahead",
        "anticipation_relative_to": "event_start_time_idx (NOT the specific outbreak week being predicted)",
    }

    # strict
    summary.update(_summary_for("model", "model_detected", "model_max_anticipation_weeks", ""))
    summary.update(_summary_for("naive", "naive_detected", "naive_max_anticipation_weeks", ""))

    # relaxed
    summary.update(_summary_for("model", "model_detected_relaxed", "model_max_anticipation_weeks_relaxed", "_relaxed"))
    summary.update(_summary_for("naive", "naive_detected_relaxed", "naive_max_anticipation_weeks_relaxed", "_relaxed"))

    return event_level, pd.DataFrame([summary])

def build_error_strata_report(pred_df: pd.DataFrame, model_name: str, fold_name: str, partition_name: str) -> pd.DataFrame:
    df_eval = pred_df.copy()
    strata = {
        "all_obs": np.ones(len(df_eval), dtype=bool),
        "true_zero": (df_eval["y_true"] == 0).values,
        "true_positive_non_outbreak": ((df_eval["y_true"] > 0) & (df_eval["is_outbreak_true"] == 0)).values,
        "true_outbreak": (df_eval["is_outbreak_true"] == 1).values,
    }

    rows = []
    for name, mask in strata.items():
        mask = np.asarray(mask, dtype=bool)
        g = df_eval.loc[mask].copy()
        if len(g) == 0:
            rows.append({
                "model_name": model_name,
                "fold_name": fold_name,
                "partition_name": partition_name,
                "stratum": name,
                "n_obs": 0,
                "model_MAE": np.nan,
                "model_RMSE": np.nan,
                "model_sMAPE": np.nan,
                "mean_true": np.nan,
                "mean_model_pred": np.nan,
            })
            continue

        y = g["y_true"].values
        yhat = g["y_pred"].values

        model_mae = float(mean_absolute_error(y, yhat))
        model_rmse = float(np.sqrt(mean_squared_error(y, yhat)))
        model_smape = float(smape(y, yhat))

        rows.append({
            "model_name": model_name,
            "fold_name": fold_name,
            "partition_name": partition_name,
            "stratum": name,
            "n_obs": int(len(g)),
            "model_MAE": model_mae,
            "model_RMSE": model_rmse,
            "model_sMAPE": model_smape,
            "mean_true": float(np.mean(y)),
            "mean_model_pred": float(np.mean(yhat)),
        })
    return pd.DataFrame(rows)

# =========================
# 12. EXCEL WRITER
# =========================
def write_excel_report(excel_path: Path, sheets: Dict[str, pd.DataFrame]):
    ensure_dir(excel_path.parent)
    with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
        for sheet_name in SHEET_ORDER:
            df_sheet = sheets.get(sheet_name, None)
            clean_name = sanitize_sheet_name(sheet_name)
            if df_sheet is None:
                continue
            if not isinstance(df_sheet, pd.DataFrame):
                df_sheet = pd.DataFrame(df_sheet)
            df_sheet.to_excel(writer, sheet_name=clean_name, index=False)

        # Write any extra sheets not in the predefined order at the end
        for sheet_name, df_sheet in sheets.items():
            if sheet_name in SHEET_ORDER:
                continue
            clean_name = sanitize_sheet_name(sheet_name)
            if not isinstance(df_sheet, pd.DataFrame):
                df_sheet = pd.DataFrame(df_sheet)
            df_sheet.to_excel(writer, sheet_name=clean_name, index=False)

# =========================
# 13. COMMON FOLD PREPARATION
# =========================
STATIC_REAL_COLS = ["mun_std_cases_hist", "mun_positive_rate_hist"]

def build_feature_lists():
    known_real_cols = ["week_in_year", "week_sin", "week_cos"]
    unknown_real_cols = (
        [TARGET_COL, FLOW_COL]
        + AGE_COLS
        + [f"{TARGET_COL}_lag{lag}" for lag in TARGET_LAGS]
        + [f"{TARGET_COL}_roll{w}" for w in TARGET_ROLLS]
        + [f"{FLOW_COL}_lag{lag}" for lag in FLOW_LAGS]
        + [f"{FLOW_COL}_roll{w}" for w in FLOW_ROLLS]
    )

    if FUTURE_CLIMATE_IS_TRULY_KNOWN:
        known_real_cols += CLIMATE_COLS
    else:
        unknown_real_cols += CLIMATE_COLS

    return known_real_cols, unknown_real_cols

def common_dynamic_feature_cols():
    known_real_cols, unknown_real_cols = build_feature_lists()
    dynamic = []
    dynamic += known_real_cols
    dynamic += [c for c in unknown_real_cols if c != TARGET_COL]
    dynamic = list(dict.fromkeys(dynamic))
    return dynamic

def build_fold_spec(
    df_model: pd.DataFrame,
    fold_name: str,
    eval_start_year: int,
    partition_name: str,
    output_dir: Path,
) -> FoldSpec:
    train_df = df_model[df_model[YEAR_COL] < eval_start_year].copy()
    eval_df = df_model[df_model[YEAR_COL] <= eval_start_year].copy()

    if partition_name == "final_test":
        train_df = df_model[df_model[YEAR_COL] < TEST_YEAR].copy()
        eval_df = df_model[df_model[YEAR_COL] <= TEST_YEAR].copy()

    if USE_ACTIVE_MUNICIPALITIES:
        active_set = select_active_municipalities(train_df)
        train_df = train_df[train_df[GROUP_COL].isin(active_set)].copy()
        eval_df = eval_df[eval_df[GROUP_COL].isin(active_set)].copy()
        print(f"[{fold_name}] Municipios activos conservados: {len(active_set):,}")

    validate_dataframe(train_df, f"{fold_name}_train_raw", output_dir / "train_raw_report.json")
    validate_dataframe(eval_df, f"{fold_name}_eval_raw", output_dir / "eval_raw_report.json")
    validate_split_logic(train_df, eval_df, eval_start_year, fold_name)

    train_aug, eval_aug, weight_info = add_fold_static_features_and_weights(train_df, eval_df)

    validate_dataframe(train_aug, f"{fold_name}_train_aug", output_dir / "train_aug_report.json")
    validate_dataframe(eval_aug, f"{fold_name}_eval_aug", output_dir / "eval_aug_report.json")

    write_json(output_dir / "weight_info.json", weight_info)

    write_json(output_dir / "outbreak_threshold_info.json", {
        "outbreak_definition": (
            f"dynamic: y_t >= max({OUTBREAK_MIN_CASES_ABS}, "
            f"(1+{OUTBREAK_GROWTH_RATE}) * baseline_t), "
            f"baseline_t = mean(previous {OUTBREAK_BASELINE_WINDOW} weeks, past-only)"
        ),
        "OUTBREAK_BASELINE_WINDOW": int(OUTBREAK_BASELINE_WINDOW),
        "OUTBREAK_BASELINE_MIN_HISTORY": int(OUTBREAK_BASELINE_MIN_HISTORY),
        "OUTBREAK_GROWTH_RATE": float(OUTBREAK_GROWTH_RATE),
        "OUTBREAK_MIN_CASES_ABS": float(OUTBREAK_MIN_CASES_ABS),
        "n_municipalities_train": int(train_df[GROUP_COL].nunique()),
        "n_municipalities_eval": int(eval_df[GROUP_COL].nunique()),
    })

    return FoldSpec(
        fold_name=fold_name,
        eval_start_year=eval_start_year,
        partition_name=partition_name,
        train_df=train_aug,
        eval_df=eval_aug,
        output_dir=output_dir,
    )

def evaluation_windows_df(eval_df: pd.DataFrame, eval_start_year: int) -> pd.DataFrame:
    eval_part = eval_df[eval_df[YEAR_COL] == eval_start_year].copy()
    max_t = int(eval_df[TIME_COL].max())
    max_start_allowed = max_t - (MAX_PREDICTION_LENGTH - 1)
    starts = eval_part[[GROUP_COL, TIME_COL]].drop_duplicates().rename(columns={TIME_COL: "prediction_start_time_idx"})
    starts = starts[starts["prediction_start_time_idx"] <= max_start_allowed].copy()
    starts[GROUP_COL] = starts[GROUP_COL].astype(str)
    return starts.sort_values([GROUP_COL, "prediction_start_time_idx"]).reset_index(drop=True)

# =========================
# 14. NAIVE MODEL
# =========================
def predict_naive_common(spec: FoldSpec) -> pd.DataFrame:
    starts_df = evaluation_windows_df(spec.eval_df, spec.eval_start_year).copy()

    target_lookup = (
        spec.eval_df[[GROUP_COL, TIME_COL, TARGET_COL]]
        .drop_duplicates([GROUP_COL, TIME_COL])
        .copy()
    )
    target_lookup[GROUP_COL] = target_lookup[GROUP_COL].astype(str)

    # naive = valor observado en t-1
    naive_lookup = target_lookup.rename(
        columns={
            TIME_COL: "naive_time_idx",
            TARGET_COL: "y_pred"
        }
    )

    starts_df["naive_time_idx"] = starts_df["prediction_start_time_idx"] - 1
    starts_df = starts_df.merge(
        naive_lookup[[GROUP_COL, "naive_time_idx", "y_pred"]],
        on=[GROUP_COL, "naive_time_idx"],
        how="left"
    )
    starts_df["y_pred"] = starts_df["y_pred"].fillna(0.0).clip(lower=0.0)

    # expandir horizontes 1..4 
    horizon_df = pd.DataFrame({"horizon": np.arange(1, MAX_PREDICTION_LENGTH + 1, dtype=int)})
    starts_df["_tmp_key"] = 1
    horizon_df["_tmp_key"] = 1
    out = starts_df.merge(horizon_df, on="_tmp_key", how="left").drop(columns="_tmp_key")

    out["target_time_idx"] = out["prediction_start_time_idx"] + (out["horizon"] - 1)

    y_true_lookup = target_lookup.rename(
        columns={
            TIME_COL: "target_time_idx",
            TARGET_COL: "y_true"
        }
    )

    out = out.merge(
        y_true_lookup[[GROUP_COL, "target_time_idx", "y_true"]],
        on=[GROUP_COL, "target_time_idx"],
        how="left"
    )

    out = out.dropna(subset=["y_true"]).copy()
    out["y_true"] = out["y_true"].astype(float)
    out["y_pred"] = out["y_pred"].astype(float)

    return out[[GROUP_COL, "prediction_start_time_idx", "horizon", "target_time_idx", "y_true", "y_pred"]]

def run_naive_model(spec: FoldSpec) -> ModelArtifacts:
    model_name = "NAIVE"
    pred_df = predict_naive_common(spec)

    pred_df = prepare_predictions_for_evaluation(
        pred_df,
        source_eval_df=spec.eval_df,
    )

    overall, regression_by_horizon, positive_by_horizon, outbreak_by_horizon, outbreak_confusion_by_horizon, metrics_by_municipality = compute_overall_and_horizon_metrics(
        pred_df=pred_df,
        model_name=model_name,
        fold_name=spec.fold_name,
        partition_name=spec.partition_name,
    )

    event_level, anticipation_summary = build_outbreak_anticipation_report(pred_df)
    error_strata = build_error_strata_report(pred_df, model_name, spec.fold_name, spec.partition_name)

    return ModelArtifacts(
        model_name=model_name,
        fold_name=spec.fold_name,
        partition_name=spec.partition_name,
        overall=overall,
        regression_by_horizon=regression_by_horizon,
        positive_by_horizon=positive_by_horizon,
        outbreak_by_horizon=outbreak_by_horizon,
        outbreak_confusion_by_horizon=outbreak_confusion_by_horizon,
        anticipation_summary=anticipation_summary,
        anticipation_event_level=event_level,
        error_strata=error_strata,
        predictions=pred_df,
        training_history=pd.DataFrame(),
        top_checkpoints=pd.DataFrame(),
        metrics_by_municipality=metrics_by_municipality,
        metadata={"notes": "Naive lag-1 direct baseline"},
    )

# =========================
# 15. ARIMA MODEL
# =========================
def select_arima_order(
    train_series: pd.Series,
    muni: str = "unknown"
) -> Tuple[Tuple[int, int, int], Dict[str, Any]]:
    y = train_series.astype(float).values

    if not PMDARIMA_AVAILABLE:
        raise ImportError("pmdarima no está disponible. No se puede usar auto_arima.")

    if not ARIMA_USE_AUTO_ARIMA:
        raise ValueError(
            "ARIMA_USE_AUTO_ARIMA está en False. Cambiar a TRUE."
        )

    model = auto_arima(
        y,
        start_p=0, start_q=0,
        max_p=4, max_q=4,
        d=None, max_d=1,
        seasonal=False,
        stepwise=True,
        suppress_warnings=True,
        error_action="ignore",
        trace=False,
        n_fits=20,
        with_intercept=True,
    )

    order = tuple(model.order)

    meta = {
        "method": "auto_arima_non_seasonal",
        "aic": float(getattr(model, "aic", lambda: np.nan)()),
        "auto_arima_error": None,
    }

    print(f"[ARIMA][{muni}] auto_arima OK | order={order}")
    return order, meta


def arima_forecast_for_municipality(
    muni_df_train: pd.DataFrame,
    muni_df_eval: pd.DataFrame,
    eval_start_year: int,
    allowed_start_times: Optional[List[int]] = None,
) -> Tuple[pd.DataFrame, Dict[str, Any]]:
    muni = str(muni_df_train[GROUP_COL].iloc[0]) if len(muni_df_train) > 0 else str(muni_df_eval[GROUP_COL].iloc[0])

    train_series_df = (
        muni_df_train[[TIME_COL, TARGET_COL]]
        .drop_duplicates()
        .sort_values(TIME_COL)
        .copy()
    )

    eval_series_df = (
        muni_df_eval[[TIME_COL, YEAR_COL, TARGET_COL]]
        .drop_duplicates()
        .sort_values(TIME_COL)
        .copy()
    )
    eval_series_df[GROUP_COL] = muni

    if allowed_start_times is None:
        eval_target_df = eval_series_df[eval_series_df[YEAR_COL] == eval_start_year].copy()
        eval_start_times = sorted(eval_target_df[TIME_COL].astype(int).unique())
    else:
        eval_start_times = sorted([int(x) for x in allowed_start_times])

    if len(eval_start_times) == 0:
        return pd.DataFrame(columns=[GROUP_COL, "prediction_start_time_idx", "horizon", "target_time_idx", "y_true", "y_pred"]), {
            "muni": muni,
            "order": None,
            "order_selection_method": None,
            "order_selection_aic": np.nan,
            "failed_fit": 0,
            "n_eval_points": 0,
            "n_windows": 0,
        }

    full_hist_df = pd.concat([
        train_series_df[[TIME_COL, TARGET_COL]],
        eval_series_df[[TIME_COL, TARGET_COL]]
    ], axis=0).drop_duplicates(subset=[TIME_COL]).sort_values(TIME_COL).reset_index(drop=True)

    train_values = train_series_df[TARGET_COL].astype(float).values
    if len(train_values) > ARIMA_MAX_HISTORY:
        train_values_for_order = train_values[-ARIMA_MAX_HISTORY:]
    else:
        train_values_for_order = train_values.copy()

    if len(train_values_for_order) < 8:
        order = None
        order_meta = {"method": "fallback_short_history", "aic": np.nan}
    else:
        order, order_meta = select_arima_order(pd.Series(train_values_for_order), muni=muni)

    rows = []
    failed_fit_count = 0
    n_windows = 0

    full_hist_lookup = full_hist_df.set_index(TIME_COL)[TARGET_COL].astype(float)

    for start_idx in eval_start_times:
        n_windows += 1

        hist = full_hist_lookup.loc[full_hist_lookup.index < start_idx].sort_index().values.astype(float)
        if len(hist) > ARIMA_MAX_HISTORY:
            hist = hist[-ARIMA_MAX_HISTORY:]

        future_rows = eval_series_df[
            (eval_series_df[TIME_COL] >= start_idx) &
            (eval_series_df[TIME_COL] <= start_idx + MAX_PREDICTION_LENGTH - 1)
        ].sort_values(TIME_COL).copy()

        # Como ahora usamos starts válidos comunes, aquí deberían existir exactamente 4 pasos
        if len(future_rows) != MAX_PREDICTION_LENGTH:
            raise ValueError(
                f"[ARIMA][{muni}] Ventana incompleta detectada pese al filtrado común. "
                f"start_idx={start_idx}, len(future_rows)={len(future_rows)}"
            )

        if (order is None) or (len(hist) < 8):
            fallback = float(hist[-1]) if len(hist) > 0 else 0.0
            preds = np.repeat(fallback, len(future_rows)).astype(float)
            failed_fit_count += 1
        else:
            try:
                d = int(order[1])
                trend_arg = "c" if d == 0 else "n"

                model = ARIMA(
                    hist,
                    order=order,
                    trend=trend_arg,
                )
                res = model.fit()
                preds = np.asarray(res.forecast(steps=len(future_rows)), dtype=float)
                preds = np.clip(preds, 0.0, None)
            except Exception as e:
                print(f"[ARIMA][{muni}] rolling fit FAILED | start_idx={start_idx} | order={order} | trend={trend_arg} | error={repr(e)}")
                fallback = float(hist[-1]) if len(hist) > 0 else 0.0
                preds = np.repeat(fallback, len(future_rows)).astype(float)
                failed_fit_count += 1

        for j, (_, fut_row) in enumerate(future_rows.iterrows(), start=1):
            target_idx = int(fut_row[TIME_COL])
            h = target_idx - start_idx + 1

            rows.append({
                GROUP_COL: muni,
                "prediction_start_time_idx": int(start_idx),
                "horizon": int(h),
                "target_time_idx": int(target_idx),
                "y_true": float(fut_row[TARGET_COL]),
                "y_pred": float(preds[j - 1]),
            })

    meta = {
        "muni": muni,
        "order": order,
        "order_selection_method": order_meta["method"],
        "order_selection_aic": order_meta["aic"],
        "failed_fit": int(failed_fit_count),
        "n_eval_points": int(len(eval_start_times)),
        "n_windows": int(n_windows),
    }

    return pd.DataFrame(rows), meta

def run_arima_model(spec: FoldSpec) -> ModelArtifacts:
    model_name = "ARIMA"
    pred_parts = []
    meta_rows = []

    train_grp = {str(k): g.copy() for k, g in spec.train_df.groupby(GROUP_COL)}
    eval_grp = {str(k): g.copy() for k, g in spec.eval_df.groupby(GROUP_COL)}

    common_starts_df = evaluation_windows_df(spec.eval_df, spec.eval_start_year)
    starts_by_muni = {
        str(m): sorted(g["prediction_start_time_idx"].astype(int).tolist())
        for m, g in common_starts_df.groupby(GROUP_COL)
    }

    muni_list = sorted(eval_grp.keys())
    total_munis = len(muni_list)

    print(f"[{spec.fold_name}] ARIMA | municipios a procesar: {total_munis:,}")

    for i, muni in enumerate(muni_list, start=1):
        muni_train = train_grp.get(muni, spec.train_df.iloc[0:0].copy())
        muni_eval = eval_grp[muni]

        muni_train = muni_train.sort_values(TIME_COL)
        muni_eval = muni_eval.sort_values(TIME_COL)

        allowed_start_times = starts_by_muni.get(muni, [])

        pred_m, meta_m = arima_forecast_for_municipality(
            muni_df_train=muni_train,
            muni_df_eval=muni_eval,
            eval_start_year=spec.eval_start_year,
            allowed_start_times=allowed_start_times,
        )

        if len(pred_m) > 0:
            pred_parts.append(pred_m)
        meta_rows.append(meta_m)

        if (i % 5 == 0) or (i == total_munis):
            print(
                f"[{spec.fold_name}] ARIMA | municipios procesados: "
                f"{i}/{total_munis} | último municipio: {muni} | "
                f"failed_fit acumulados: {sum(int(x.get('failed_fit', 0)) for x in meta_rows)}"
            )

    pred_df = pd.concat(pred_parts, ignore_index=True) if len(pred_parts) > 0 else pd.DataFrame(
        columns=[GROUP_COL, "prediction_start_time_idx", "horizon", "target_time_idx", "y_true", "y_pred"]
    )

    pred_df = prepare_predictions_for_evaluation(
        pred_df,
        source_eval_df=spec.eval_df,
    )

    overall, regression_by_horizon, positive_by_horizon, outbreak_by_horizon, outbreak_confusion_by_horizon, metrics_by_municipality = compute_overall_and_horizon_metrics(
        pred_df=pred_df,
        model_name=model_name,
        fold_name=spec.fold_name,
        partition_name=spec.partition_name,
    )

    event_level, anticipation_summary = build_outbreak_anticipation_report(pred_df)
    error_strata = build_error_strata_report(
        pred_df, model_name, spec.fold_name, spec.partition_name
    )

    return ModelArtifacts(
        model_name=model_name,
        fold_name=spec.fold_name,
        partition_name=spec.partition_name,
        overall=overall,
        regression_by_horizon=regression_by_horizon,
        positive_by_horizon=positive_by_horizon,
        outbreak_by_horizon=outbreak_by_horizon,
        outbreak_confusion_by_horizon=outbreak_confusion_by_horizon,
        anticipation_summary=anticipation_summary,
        anticipation_event_level=event_level,
        error_strata=error_strata,
        predictions=pred_df,
        training_history=pd.DataFrame(),
        top_checkpoints=pd.DataFrame(),
        metrics_by_municipality=metrics_by_municipality,
        metadata={"municipality_orders": pd.DataFrame(meta_rows).to_dict(orient="records")},
    )
# =========================
# 16. LSTM DATASET + MODEL
# =========================
class DengueLSTMDataset(Dataset):
    def __init__(
        self,
        df_full: pd.DataFrame,
        prediction_starts: pd.DataFrame,
        group_to_idx: Dict[str, int],
        feature_cols: List[str],
        static_cols: List[str],
        weight_col: str = "loss_weight",
    ):
        self.df_full = df_full.sort_values([GROUP_COL, TIME_COL]).copy()
        self.feature_cols = feature_cols
        self.static_cols = static_cols
        self.group_to_idx = group_to_idx
        self.weight_col = weight_col

        self.samples = []
        self.lookup = {}
        for muni, g in self.df_full.groupby(GROUP_COL):
            g = g.sort_values(TIME_COL).reset_index(drop=True)
            self.lookup[str(muni)] = g

        for _, row in prediction_starts.iterrows():
            muni = str(row[GROUP_COL])
            start_idx = int(row["prediction_start_time_idx"])
            g = self.lookup.get(muni, None)
            if g is None:
                continue

            enc = g[(g[TIME_COL] >= start_idx - MAX_ENCODER_LENGTH) & (g[TIME_COL] <= start_idx - 1)].copy()
            dec = g[(g[TIME_COL] >= start_idx) & (g[TIME_COL] <= start_idx + MAX_PREDICTION_LENGTH - 1)].copy()

            if len(enc) != MAX_ENCODER_LENGTH:
                continue
            if len(dec) != MAX_PREDICTION_LENGTH:
                continue

            self.samples.append((muni, start_idx))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        muni, start_idx = self.samples[idx]
        g = self.lookup[muni]

        enc = g[(g[TIME_COL] >= start_idx - MAX_ENCODER_LENGTH) & (g[TIME_COL] <= start_idx - 1)].copy()
        dec = g[(g[TIME_COL] >= start_idx) & (g[TIME_COL] <= start_idx + MAX_PREDICTION_LENGTH - 1)].copy()

        x_seq = enc[self.feature_cols].astype(float).values
        x_static = enc[self.static_cols].iloc[-1].astype(float).values
        y = dec[TARGET_COL].astype(float).values
        y_log = np.log1p(y)

        w = float(dec[self.weight_col].astype(float).mean())

        meta = {
            GROUP_COL: muni,
            "prediction_start_time_idx": start_idx,
            "target_time_idx": dec[TIME_COL].astype(int).values,
        }

        return {
            "x_seq": torch.tensor(x_seq, dtype=torch.float32),
            "x_static": torch.tensor(x_static, dtype=torch.float32),
            "muni_idx": torch.tensor(self.group_to_idx[muni], dtype=torch.long),
            "y_log": torch.tensor(y_log, dtype=torch.float32),
            "y_true": torch.tensor(y, dtype=torch.float32),
            "loss_weight": torch.tensor(w, dtype=torch.float32),
            "meta": meta,
        }

def lstm_collate_fn(batch):
    x_seq = torch.stack([b["x_seq"] for b in batch], dim=0)
    x_static = torch.stack([b["x_static"] for b in batch], dim=0)
    muni_idx = torch.stack([b["muni_idx"] for b in batch], dim=0)
    y_log = torch.stack([b["y_log"] for b in batch], dim=0)
    y_true = torch.stack([b["y_true"] for b in batch], dim=0)
    loss_weight = torch.stack([b["loss_weight"] for b in batch], dim=0)

    meta_group = [b["meta"][GROUP_COL] for b in batch]
    meta_start = [b["meta"]["prediction_start_time_idx"] for b in batch]
    meta_target = [b["meta"]["target_time_idx"] for b in batch]

    return {
        "x_seq": x_seq,
        "x_static": x_static,
        "muni_idx": muni_idx,
        "y_log": y_log,
        "y_true": y_true,
        "loss_weight": loss_weight,
        "meta_group": meta_group,
        "meta_start": meta_start,
        "meta_target": meta_target,
    }

class LSTMMultiHorizon(pl.LightningModule):
    def __init__(
        self,
        n_features: int,
        n_static: int,
        n_municipalities: int,
        learning_rate: float = LSTM_LEARNING_RATE,
        use_plateau_scheduler: bool = True, 
    ):
        super().__init__()
        self.save_hyperparameters()
        self.use_plateau_scheduler = use_plateau_scheduler

        self.muni_emb = nn.Embedding(n_municipalities, LSTM_EMBED_DIM)
        self.static_proj = nn.Sequential(
            nn.Linear(n_static + LSTM_EMBED_DIM, LSTM_STATIC_HIDDEN),
            nn.ReLU(),
            nn.Dropout(LSTM_DROPOUT),
        )

        self.lstm = nn.LSTM(
            input_size=n_features,
            hidden_size=LSTM_HIDDEN_SIZE,
            num_layers=LSTM_NUM_LAYERS,
            dropout=LSTM_DROPOUT if LSTM_NUM_LAYERS > 1 else 0.0,
            batch_first=True,
        )

        self.head = nn.Sequential(
            nn.Linear(LSTM_HIDDEN_SIZE + LSTM_STATIC_HIDDEN, 128),
            nn.ReLU(),
            nn.Dropout(LSTM_DROPOUT),
            nn.Linear(128, MAX_PREDICTION_LENGTH),
        )

        self.loss_fn = nn.SmoothL1Loss(reduction="none")
        self.learning_rate = learning_rate

    def forward(self, x_seq, x_static, muni_idx):
        lstm_out, (h_n, c_n) = self.lstm(x_seq)
        h_last = h_n[-1]

        muni_emb = self.muni_emb(muni_idx)
        static_input = torch.cat([x_static, muni_emb], dim=1)
        static_repr = self.static_proj(static_input)

        fused = torch.cat([h_last, static_repr], dim=1)
        pred_log = self.head(fused)
        return pred_log

    def _weighted_loss(self, pred_log, y_log, loss_weight):
        raw = self.loss_fn(pred_log, y_log).mean(dim=1)
        return (raw * loss_weight).mean()

    def training_step(self, batch, batch_idx):
        pred_log = self(batch["x_seq"], batch["x_static"], batch["muni_idx"])
        loss = self._weighted_loss(pred_log, batch["y_log"], batch["loss_weight"])
        self.log("train_loss", loss, prog_bar=True, on_step=False, on_epoch=True)
        return loss

    def validation_step(self, batch, batch_idx):
        pred_log = self(batch["x_seq"], batch["x_static"], batch["muni_idx"])
        loss = self._weighted_loss(pred_log, batch["y_log"], batch["loss_weight"])
        self.log("val_loss", loss, prog_bar=True, on_step=False, on_epoch=True)
        return loss

    def predict_step(self, batch, batch_idx, dataloader_idx=0):
        pred_log = self(batch["x_seq"], batch["x_static"], batch["muni_idx"])
        pred = torch.expm1(pred_log).clamp(min=0.0)
        return {
            "pred": pred.detach().cpu(),
            "y_true": batch["y_true"].detach().cpu(),
            "meta_group": batch["meta_group"],
            "meta_start": batch["meta_start"],
            "meta_target": batch["meta_target"],
        }

    def configure_optimizers(self):
        opt = torch.optim.Adam(self.parameters(), lr=self.learning_rate)
        # Usar ReduceLROnPlateau solo si se proporciona una métrica de validación
        # Se espera que el trainer pase el monitor si está disponible
        if getattr(self, 'use_plateau_scheduler', True):
            scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
                opt, mode="min", factor=0.5, patience=3
            )
            return {
                "optimizer": opt,
                "lr_scheduler": {
                    "scheduler": scheduler,
                    "monitor": "val_loss",
                },
            }
        else:
            # Scheduler simple sin necesidad de monitor (por ejemplo, StepLR cada 10 epochs)
            scheduler = torch.optim.lr_scheduler.StepLR(opt, step_size=10, gamma=0.9)
            return {
                "optimizer": opt,
                "lr_scheduler": {
                    "scheduler": scheduler,
                    "monitor": None,  # No se necesita monitor
                },
            }

# =========================
# 17. LSTM HELPERS
# =========================
def build_lstm_feature_matrix_columns() -> Tuple[List[str], List[str]]:
    feature_cols = [TARGET_COL] + common_dynamic_feature_cols()
    feature_cols = list(dict.fromkeys(feature_cols))
    static_cols = STATIC_REAL_COLS
    return feature_cols, static_cols

def build_lstm_group_index(df_train: pd.DataFrame, df_eval: pd.DataFrame) -> Dict[str, int]:
    munis = sorted(set(df_train[GROUP_COL].astype(str).unique()).union(set(df_eval[GROUP_COL].astype(str).unique())))
    return {m: i for i, m in enumerate(munis)}

def build_lstm_dataloaders(spec: FoldSpec):
    feature_cols, static_cols = build_lstm_feature_matrix_columns()
    group_to_idx = build_lstm_group_index(spec.train_df, spec.eval_df)

    # Train windows inside train only
    train_starts = []
    for muni, g in spec.train_df.groupby(GROUP_COL):
        g = g.sort_values(TIME_COL)
        min_start = int(g[TIME_COL].min()) + MAX_ENCODER_LENGTH
        max_start = int(g[TIME_COL].max()) - (MAX_PREDICTION_LENGTH - 1)
        for s in range(min_start, max_start + 1):
            train_starts.append({GROUP_COL: str(muni), "prediction_start_time_idx": s})
    train_starts = pd.DataFrame(train_starts)

    # Eval windows start only from eval year
    val_starts = evaluation_windows_df(spec.eval_df, spec.eval_start_year)

    train_ds = DengueLSTMDataset(
        df_full=spec.train_df,
        prediction_starts=train_starts,
        group_to_idx=group_to_idx,
        feature_cols=feature_cols,
        static_cols=static_cols,
    )
    val_ds = DengueLSTMDataset(
        df_full=spec.eval_df,
        prediction_starts=val_starts,
        group_to_idx=group_to_idx,
        feature_cols=feature_cols,
        static_cols=static_cols,
    )

    g_loader = torch.Generator()
    g_loader.manual_seed(SEED)
    train_loader = DataLoader(
        train_ds,
        batch_size=LSTM_BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        collate_fn=lstm_collate_fn,
        generator=g_loader,
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=LSTM_BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        collate_fn=lstm_collate_fn,
    )
    meta = {
        "feature_cols": feature_cols,
        "static_cols": static_cols,
        "n_features": len(feature_cols),
        "n_static": len(static_cols),
        "n_municipalities": len(group_to_idx),
        "n_train_samples": len(train_ds),
        "n_eval_samples": len(val_ds),
    }
    return train_loader, val_loader, group_to_idx, meta

def collect_lstm_predictions(model: LSTMMultiHorizon, loader: DataLoader) -> pd.DataFrame:
    trainer = pl.Trainer(
        logger=False,
        enable_checkpointing=False,
        enable_progress_bar=False,
        enable_model_summary=False,
        accelerator="cpu",
        devices=1,
        deterministic=True,
    )

    pred_batches = trainer.predict(model, dataloaders=loader)

    rows = []
    for batch in pred_batches:
        pred = batch["pred"].numpy()
        y_true = batch["y_true"].numpy()
        meta_group = batch["meta_group"]
        meta_start = batch["meta_start"]
        meta_target = batch["meta_target"]

        n = pred.shape[0]
        for i in range(n):
            muni = str(meta_group[i])
            start_idx = int(meta_start[i])
            target_idxs = meta_target[i]

            for h in range(1, MAX_PREDICTION_LENGTH + 1):
                rows.append({
                    GROUP_COL: muni,
                    "prediction_start_time_idx": start_idx,
                    "horizon": h,
                    "target_time_idx": int(target_idxs[h - 1]),
                    "y_true": float(y_true[i, h - 1]),
                    "y_pred": float(max(pred[i, h - 1], 0.0)),
                })

    return pd.DataFrame(rows)

def checkpoint_epoch_1based(ckpt_path: str) -> int:
    state = torch.load(ckpt_path, map_location="cpu")
    epoch0 = int(state.get("epoch", 0))
    return epoch0 + 1

def get_sorted_top_checkpoints(checkpoint_callback: ModelCheckpoint) -> pd.DataFrame:
    rows = []
    best_k_models = getattr(checkpoint_callback, "best_k_models", {})
    for path, score in best_k_models.items():
        try:
            score_val = float(score.cpu().item()) if hasattr(score, "cpu") else float(score)
        except Exception:
            score_val = np.nan
        rows.append({
            "checkpoint_path": str(path),
            "val_loss": score_val,
            "checkpoint_epoch_1based": checkpoint_epoch_1based(str(path)) if path else np.nan,
        })
    df_ckpts = pd.DataFrame(rows)
    if len(df_ckpts) == 0:
        return pd.DataFrame(columns=["rank", "checkpoint_path", "val_loss", "checkpoint_epoch_1based"])
    df_ckpts = df_ckpts.sort_values(["val_loss", "checkpoint_epoch_1based"], ascending=[True, True]).reset_index(drop=True)
    df_ckpts.insert(0, "rank", np.arange(1, len(df_ckpts) + 1))
    return df_ckpts

class EpochPrinterCallback(Callback):
    def __init__(self, fold_name="fold"):
        super().__init__()
        self.fold_name = fold_name

    def on_validation_epoch_end(self, trainer, pl_module):
        if getattr(trainer, "sanity_checking", False):
            return
        metrics = trainer.callback_metrics
        epoch = trainer.current_epoch
        train_loss = metrics.get("train_loss_epoch", metrics.get("train_loss"))
        val_loss = metrics.get("val_loss")
        lr = None
        if trainer.optimizers and len(trainer.optimizers) > 0:
            lr = trainer.optimizers[0].param_groups[0]["lr"]

        def _to_float(x):
            if x is None:
                return None
            if hasattr(x, "detach"):
                return float(x.detach().cpu().item())
            try:
                return float(x)
            except Exception:
                return None

        print(
            f"[{self.fold_name}] epoch={epoch:03d} | "
            f"train_loss={_to_float(train_loss)} | "
            f"val_loss={_to_float(val_loss)} | "
            f"lr={_to_float(lr)}"
        )

def run_lstm_model(spec: FoldSpec, refit_epochs: Optional[int] = None) -> ModelArtifacts:
    model_name = "LSTM"
    train_loader, val_loader, group_to_idx, meta = build_lstm_dataloaders(spec)

    logger = CSVLogger(
        save_dir=str(spec.output_dir / model_name.lower()),
        name="",
        version="logs",
        flush_logs_every_n_steps=10,
    )
    ckpt_dir = spec.output_dir / model_name.lower() / "_temp_ckpt"
    ensure_dir(ckpt_dir)

    use_eval_for_model_selection = (spec.partition_name != "final_test")

    if use_eval_for_model_selection:
        checkpoint_callback = ModelCheckpoint(
            dirpath=str(ckpt_dir),
            filename="best-epoch{epoch:03d}-val{val_loss:.4f}",
            monitor="val_loss",
            mode="min",
            save_top_k=TOP_K_CKPTS,
            save_last=True,
        )
        callbacks = [
            checkpoint_callback,
            EarlyStopping(monitor="val_loss", patience=5, min_delta=1e-3, mode="min"),
            LearningRateMonitor(logging_interval="epoch"),
            EpochPrinterCallback(fold_name=f"{spec.fold_name}_{model_name}"),
        ]
    else:
        checkpoint_callback = ModelCheckpoint(
            dirpath=str(ckpt_dir),
            filename="last-epoch{epoch:03d}",
            save_top_k=0,
            save_last=True,
            every_n_epochs=1,
        )
        callbacks = [
            checkpoint_callback,
            LearningRateMonitor(logging_interval="epoch"),
            EpochPrinterCallback(fold_name=f"{spec.fold_name}_{model_name}"),
        ]

    reseed_for_run(f"{spec.fold_name}|LSTM|model_init_and_train")
    model = LSTMMultiHorizon(
        n_features=meta["n_features"],
        n_static=meta["n_static"],
        n_municipalities=meta["n_municipalities"],
        learning_rate=LSTM_LEARNING_RATE,
        use_plateau_scheduler=use_eval_for_model_selection,
    )

    trainer = pl.Trainer(
        max_epochs=refit_epochs if refit_epochs is not None else LSTM_MAX_EPOCHS,
        gradient_clip_val=LSTM_GRAD_CLIP,
        logger=logger,
        callbacks=callbacks,
        log_every_n_steps=10,
        num_sanity_val_steps=0,
        enable_progress_bar=True,
        enable_model_summary=False,
        deterministic=True,
        **get_trainer_accelerator(),
    )

    last_ckpt = ckpt_dir / "last.ckpt"
    resume_ckpt = str(last_ckpt) if last_ckpt.exists() else None

    if use_eval_for_model_selection:
        trainer.fit(
            model,
            train_dataloaders=train_loader,
            val_dataloaders=val_loader,
            ckpt_path=resume_ckpt,
        )

        selected_ckpt_path = checkpoint_callback.best_model_path
        if selected_ckpt_path is None or str(selected_ckpt_path).strip() == "":
            selected_ckpt_path = str(last_ckpt) if last_ckpt.exists() else None
        if selected_ckpt_path is None:
            raise RuntimeError(f"[{spec.fold_name}] No se encontró checkpoint LSTM válido.")

        selected_model = LSTMMultiHorizon.load_from_checkpoint(selected_ckpt_path)
        selected_epoch_1based = checkpoint_epoch_1based(selected_ckpt_path)
        top_ckpts_df = get_sorted_top_checkpoints(checkpoint_callback)

    else:
        trainer.fit(
            model,
            train_dataloaders=train_loader,
            ckpt_path=resume_ckpt,
        )

        selected_ckpt_path = str(last_ckpt) if last_ckpt.exists() else None

        if selected_ckpt_path is not None:
            selected_model = LSTMMultiHorizon.load_from_checkpoint(selected_ckpt_path)
            selected_epoch_1based = checkpoint_epoch_1based(selected_ckpt_path)
        else:
            selected_model = model
            selected_epoch_1based = int(refit_epochs if refit_epochs is not None else LSTM_MAX_EPOCHS)

        top_ckpts_df = pd.DataFrame(
            columns=["rank", "checkpoint_path", "val_loss", "checkpoint_epoch_1based"]
        )

    pred_df = collect_lstm_predictions(selected_model, val_loader)

    pred_df = prepare_predictions_for_evaluation(
        pred_df,
        source_eval_df=spec.eval_df,
    )

    overall, regression_by_horizon, positive_by_horizon, outbreak_by_horizon, outbreak_confusion_by_horizon, metrics_by_municipality = compute_overall_and_horizon_metrics(
        pred_df=pred_df,
        model_name=model_name,
        fold_name=spec.fold_name,
        partition_name=spec.partition_name,
    )

    event_level, anticipation_summary = build_outbreak_anticipation_report(pred_df)
    error_strata = build_error_strata_report(pred_df, model_name, spec.fold_name, spec.partition_name)

    log_metrics_path = Path(logger.log_dir) / "metrics.csv"
    training_history_df = pd.read_csv(log_metrics_path) if log_metrics_path.exists() else pd.DataFrame()

    overall["selected_checkpoint_epoch_1based"] = int(selected_epoch_1based)
    overall["selected_checkpoint_path"] = (
        selected_ckpt_path if selected_ckpt_path is not None else "in_memory_final_weights"
    )

    metadata = {
        **meta,
        "selected_checkpoint_path": overall["selected_checkpoint_path"],
        "selected_checkpoint_epoch_1based": overall["selected_checkpoint_epoch_1based"],
        "checkpoint_selection_mode": (
            "best_val_loss_on_eval" if use_eval_for_model_selection else "last_epoch_fixed_refit"
        ),
        "n_params": int(sum(p.numel() for p in model.parameters() if p.requires_grad)),
    }

    return ModelArtifacts(
        model_name=model_name,
        fold_name=spec.fold_name,
        partition_name=spec.partition_name,
        overall=overall,
        regression_by_horizon=regression_by_horizon,
        positive_by_horizon=positive_by_horizon,
        outbreak_by_horizon=outbreak_by_horizon,
        outbreak_confusion_by_horizon=outbreak_confusion_by_horizon,
        anticipation_summary=anticipation_summary,
        anticipation_event_level=event_level,
        error_strata=error_strata,
        predictions=pred_df,
        training_history=training_history_df,
        top_checkpoints=top_ckpts_df,
        metrics_by_municipality=metrics_by_municipality,
        metadata=metadata,
    )

# =========================
# 18. TFT MODEL
# =========================
def build_tft_datasets(train_df: pd.DataFrame, full_eval_df: pd.DataFrame, prediction_start_idx: int):
    known_real_cols, unknown_real_cols = build_feature_lists()

    training = TimeSeriesDataSet(
        train_df,
        time_idx=TIME_COL,
        target=TARGET_COL,
        group_ids=[GROUP_COL],
        weight="loss_weight",
        min_encoder_length=MIN_ENCODER_LENGTH,
        max_encoder_length=MAX_ENCODER_LENGTH,
        max_prediction_length=MAX_PREDICTION_LENGTH,
        static_categoricals=[GROUP_COL],
        static_reals=STATIC_REAL_COLS,
        time_varying_known_reals=known_real_cols,
        time_varying_unknown_reals=unknown_real_cols,
        add_relative_time_idx=True,
        add_target_scales=True,
        allow_missing_timesteps=False,
        target_normalizer=GroupNormalizer(groups=[GROUP_COL], center=False, transformation="log1p"),
    )

    evaluation = TimeSeriesDataSet.from_dataset(
        training,
        full_eval_df,
        min_prediction_idx=prediction_start_idx,
        stop_randomization=True,
    )
    return training, evaluation

def build_training_only_tft_dataset(train_df: pd.DataFrame):
    known_real_cols, unknown_real_cols = build_feature_lists()

    training = TimeSeriesDataSet(
        train_df,
        time_idx=TIME_COL,
        target=TARGET_COL,
        group_ids=[GROUP_COL],
        weight="loss_weight",
        min_encoder_length=MIN_ENCODER_LENGTH,
        max_encoder_length=MAX_ENCODER_LENGTH,
        max_prediction_length=MAX_PREDICTION_LENGTH,
        static_categoricals=[GROUP_COL],
        static_reals=STATIC_REAL_COLS,
        time_varying_known_reals=known_real_cols,
        time_varying_unknown_reals=unknown_real_cols,
        add_relative_time_idx=True,
        add_target_scales=True,
        allow_missing_timesteps=False,
        target_normalizer=GroupNormalizer(groups=[GROUP_COL], center=False, transformation="log1p"),
    )
    return training

def make_tft_from_dataset(dataset, learning_rate=TFT_LEARNING_RATE, reduce_on_plateau_patience=4):
    tft = TemporalFusionTransformer.from_dataset(
        dataset,
        learning_rate=learning_rate,
        hidden_size=TFT_HIDDEN_SIZE,
        attention_head_size=TFT_ATTENTION_HEAD_SIZE,
        dropout=TFT_DROPOUT,
        hidden_continuous_size=TFT_HIDDEN_CONT_SIZE,
        loss=NegativeBinomialDistributionLoss(),
        output_size=2,
        log_interval=10,
        reduce_on_plateau_patience=reduce_on_plateau_patience,   # <-- NUEVO
    )
    return tft

def _extract_prediction_arrays(pred_obj):
    pred_output = getattr(pred_obj, "output", None)
    if pred_output is None:
        pred_output = getattr(pred_obj, "prediction", None)
    if pred_output is None:
        raise ValueError("No se encontró predicción en pred_obj.output ni pred_obj.prediction")

    pred_y = getattr(pred_obj, "y", None)
    pred_index = getattr(pred_obj, "index", None)
    y_true = pred_y[0] if isinstance(pred_y, (tuple, list)) else pred_y

    y_pred = pred_output.detach().cpu().numpy()
    y_true = y_true.detach().cpu().numpy()

    if y_pred.ndim == 3 and y_pred.shape[-1] == 1:
        y_pred = y_pred[..., 0]
    if y_true.ndim == 3 and y_true.shape[-1] == 1:
        y_true = y_true[..., 0]

    if not isinstance(pred_index, pd.DataFrame):
        pred_index = pd.DataFrame(pred_index)
    pred_index = pred_index.copy()
    pred_index[GROUP_COL] = pred_index[GROUP_COL].astype(str)
    return pred_index, y_true, y_pred

def build_prediction_frame_tft(pred_obj) -> pd.DataFrame:
    pred_index, y_true, y_pred = _extract_prediction_arrays(pred_obj)
    n_windows, pred_len = y_pred.shape

    groups = pred_index[GROUP_COL].astype(str).values
    pred_start = pred_index[TIME_COL].values.astype(int)

    out = pd.DataFrame({
        GROUP_COL: np.repeat(groups, pred_len),
        "prediction_start_time_idx": np.repeat(pred_start, pred_len),
        "horizon": np.tile(np.arange(1, pred_len + 1), n_windows),
        "target_time_idx": np.repeat(pred_start, pred_len) + np.tile(np.arange(0, pred_len), n_windows),
        "y_true": y_true.reshape(-1),
        "y_pred": y_pred.reshape(-1),
    })
    out["y_pred"] = out["y_pred"].astype(float)
    return out

def run_tft_model(spec: FoldSpec, refit_epochs: Optional[int] = None) -> ModelArtifacts:
    model_name = "TFT"
    eval_start_idx = int(spec.eval_df.loc[spec.eval_df[YEAR_COL] == spec.eval_start_year, TIME_COL].min())

    training_ds, eval_ds = build_tft_datasets(
        train_df=spec.train_df,
        full_eval_df=spec.eval_df,
        prediction_start_idx=eval_start_idx,
    )

    train_loader = training_ds.to_dataloader(
        train=True,
        batch_size=TFT_BATCH_SIZE,
        num_workers=NUM_WORKERS,
    )
    eval_loader = eval_ds.to_dataloader(
        train=False,
        batch_size=TFT_BATCH_SIZE,
        num_workers=NUM_WORKERS,
    )

    logger = CSVLogger(
        save_dir=str(spec.output_dir / model_name.lower()),
        name="",
        version="logs",
        flush_logs_every_n_steps=10,
    )
    ckpt_dir = spec.output_dir / model_name.lower() / "_temp_ckpt"
    ensure_dir(ckpt_dir)

    use_eval_for_model_selection = (spec.partition_name != "final_test")

    if use_eval_for_model_selection:
        checkpoint_callback = ModelCheckpoint(
            dirpath=str(ckpt_dir),
            filename="best-epoch{epoch:03d}-val{val_loss:.4f}",
            monitor="val_loss",
            mode="min",
            save_top_k=TOP_K_CKPTS,
            save_last=True,
        )
        callbacks = [
            checkpoint_callback,
            EarlyStopping(monitor="val_loss", patience=5, min_delta=1e-3, mode="min"),
            LearningRateMonitor(logging_interval="epoch"),
            EpochPrinterCallback(fold_name=f"{spec.fold_name}_{model_name}"),
        ]
    else:
        checkpoint_callback = ModelCheckpoint(
            dirpath=str(ckpt_dir),
            filename="last-epoch{epoch:03d}",
            save_top_k=0,
            save_last=True,
            every_n_epochs=1,
        )
        callbacks = [
            checkpoint_callback,
            LearningRateMonitor(logging_interval="epoch"),
            EpochPrinterCallback(fold_name=f"{spec.fold_name}_{model_name}"),
        ]

    reduce_patience = 4 if use_eval_for_model_selection else 0
    reseed_for_run(f"{spec.fold_name}|TFT|model_init_and_train")
    tft = make_tft_from_dataset(training_ds, reduce_on_plateau_patience=reduce_patience)

    trainer = pl.Trainer(
        max_epochs=refit_epochs if refit_epochs is not None else TFT_MAX_EPOCHS,
        gradient_clip_val=TFT_GRAD_CLIP,
        logger=logger,
        callbacks=callbacks,
        log_every_n_steps=10,
        num_sanity_val_steps=0,
        enable_progress_bar=True,
        enable_model_summary=False,
        deterministic=True,
        **get_trainer_accelerator(),
    )

    last_ckpt = ckpt_dir / "last.ckpt"
    resume_ckpt = str(last_ckpt) if last_ckpt.exists() else None

    if use_eval_for_model_selection:
        trainer.fit(
            tft,
            train_dataloaders=train_loader,
            val_dataloaders=eval_loader,
            ckpt_path=resume_ckpt,
        )

        selected_ckpt_path = checkpoint_callback.best_model_path
        if selected_ckpt_path is None or str(selected_ckpt_path).strip() == "":
            selected_ckpt_path = str(last_ckpt) if last_ckpt.exists() else None
        if selected_ckpt_path is None:
            raise RuntimeError(f"[{spec.fold_name}] No se encontró checkpoint TFT válido.")

        selected_model = TemporalFusionTransformer.load_from_checkpoint(selected_ckpt_path)
        selected_epoch_1based = checkpoint_epoch_1based(selected_ckpt_path)
        top_ckpts_df = get_sorted_top_checkpoints(checkpoint_callback)

    else:
        trainer.fit(
            tft,
            train_dataloaders=train_loader,
            ckpt_path=resume_ckpt,
        )

        selected_ckpt_path = str(last_ckpt) if last_ckpt.exists() else None

        if selected_ckpt_path is not None:
            selected_model = TemporalFusionTransformer.load_from_checkpoint(selected_ckpt_path)
            selected_epoch_1based = checkpoint_epoch_1based(selected_ckpt_path)
        else:
            selected_model = tft
            selected_epoch_1based = int(refit_epochs if refit_epochs is not None else TFT_MAX_EPOCHS)

        top_ckpts_df = pd.DataFrame(
            columns=["rank", "checkpoint_path", "val_loss", "checkpoint_epoch_1based"]
        )

    pred_obj = selected_model.predict(eval_loader, return_y=True, return_index=True)
    pred_df = build_prediction_frame_tft(pred_obj)

    pred_df = prepare_predictions_for_evaluation(
        pred_df,
        source_eval_df=spec.eval_df,
    )

    overall, regression_by_horizon, positive_by_horizon, outbreak_by_horizon, outbreak_confusion_by_horizon, metrics_by_municipality = compute_overall_and_horizon_metrics(
        pred_df=pred_df,
        model_name=model_name,
        fold_name=spec.fold_name,
        partition_name=spec.partition_name,
    )

    event_level, anticipation_summary = build_outbreak_anticipation_report(pred_df)
    error_strata = build_error_strata_report(pred_df, model_name, spec.fold_name, spec.partition_name)

    log_metrics_path = Path(logger.log_dir) / "metrics.csv"
    training_history_df = pd.read_csv(log_metrics_path) if log_metrics_path.exists() else pd.DataFrame()

    overall["selected_checkpoint_epoch_1based"] = int(selected_epoch_1based)
    overall["selected_checkpoint_path"] = (
        selected_ckpt_path if selected_ckpt_path is not None else "in_memory_final_weights"
    )

    metadata = {
        "selected_checkpoint_path": overall["selected_checkpoint_path"],
        "selected_checkpoint_epoch_1based": overall["selected_checkpoint_epoch_1based"],
        "checkpoint_selection_mode": (
            "best_val_loss_on_eval" if use_eval_for_model_selection else "last_epoch_fixed_refit"
        ),
        "n_train_sequences": int(len(training_ds)),
        "n_eval_sequences": int(len(eval_ds)),
        "n_params": int(sum(p.numel() for p in tft.parameters() if p.requires_grad)),
    }

    return ModelArtifacts(
        model_name=model_name,
        fold_name=spec.fold_name,
        partition_name=spec.partition_name,
        overall=overall,
        regression_by_horizon=regression_by_horizon,
        positive_by_horizon=positive_by_horizon,
        outbreak_by_horizon=outbreak_by_horizon,
        outbreak_confusion_by_horizon=outbreak_confusion_by_horizon,
        anticipation_summary=anticipation_summary,
        anticipation_event_level=event_level,
        error_strata=error_strata,
        predictions=pred_df,
        training_history=training_history_df,
        top_checkpoints=top_ckpts_df,
        metrics_by_municipality=metrics_by_municipality,
        metadata=metadata,
    )

# =========================
# 19. ARTIFACT PERSISTENCE
# =========================
def model_output_dir(spec: FoldSpec, model_name: str) -> Path:
    d = spec.output_dir / model_name.lower()
    ensure_dir(d)
    return d

def save_model_artifacts(spec: FoldSpec, artifacts: ModelArtifacts):
    out_dir = model_output_dir(spec, artifacts.model_name)

    summary_df = reorder_columns(pd.DataFrame([artifacts.overall]), SUMMARY_COLUMNS_ORDER)
    predictions_sample = (
        artifacts.predictions.sample(n=min(PREDICTIONS_SAMPLE_N, len(artifacts.predictions)), random_state=SEED)
        if SAVE_PREDICTIONS_SAMPLE and len(artifacts.predictions) > 0
        else pd.DataFrame()
    )

    sheets = {
        "summary_overall": summary_df,
        "regression_by_horizon": reorder_columns(artifacts.regression_by_horizon, REGRESSION_COLUMNS_ORDER),
        "positive_by_horizon": reorder_columns(artifacts.positive_by_horizon, POSITIVE_COLUMNS_ORDER),
        "outbreak_by_horizon": reorder_columns(artifacts.outbreak_by_horizon, OUTBREAK_COLUMNS_ORDER),
        "outbreak_confusion_by_horizon": artifacts.outbreak_confusion_by_horizon,
        "anticipation_summary": artifacts.anticipation_summary,
        "anticipation_event_level": artifacts.anticipation_event_level,
        "error_strata": artifacts.error_strata,
        "predictions_sample": predictions_sample,
        "training_history": artifacts.training_history if artifacts.training_history is not None else pd.DataFrame(),
        "top_checkpoints": artifacts.top_checkpoints if artifacts.top_checkpoints is not None else pd.DataFrame(),
        "predictions_metadata": pd.DataFrame([artifacts.metadata]),
    }
    if SAVE_METRICS_BY_MUNICIPALITY and artifacts.metrics_by_municipality is not None:
        sheets["metrics_by_municipality"] = artifacts.metrics_by_municipality

    excel_path = out_dir / f"{spec.fold_name}_{artifacts.model_name.lower()}_report.xlsx"
    write_excel_report(excel_path, sheets)

    summary_df.to_csv(out_dir / "summary_overall.csv", index=False)
    artifacts.regression_by_horizon.to_csv(out_dir / "regression_by_horizon.csv", index=False)
    artifacts.positive_by_horizon.to_csv(out_dir / "positive_by_horizon.csv", index=False)
    artifacts.outbreak_by_horizon.to_csv(out_dir / "outbreak_by_horizon.csv", index=False)
    artifacts.outbreak_confusion_by_horizon.to_csv(out_dir / "outbreak_confusion_by_horizon.csv", index=False)
    artifacts.anticipation_summary.to_csv(out_dir / "anticipation_summary.csv", index=False)
    artifacts.anticipation_event_level.to_csv(out_dir / "anticipation_event_level.csv", index=False)
    artifacts.error_strata.to_csv(out_dir / "error_strata.csv", index=False)
    if artifacts.training_history is not None and len(artifacts.training_history) > 0:
        artifacts.training_history.to_csv(out_dir / "training_history.csv", index=False)
    if artifacts.top_checkpoints is not None and len(artifacts.top_checkpoints) > 0:
        artifacts.top_checkpoints.to_csv(out_dir / "top_checkpoints.csv", index=False)
    if SAVE_METRICS_BY_MUNICIPALITY and artifacts.metrics_by_municipality is not None:
        artifacts.metrics_by_municipality.to_csv(out_dir / "metrics_by_municipality.csv", index=False)

    if SAVE_PREDICTIONS_FULL_PARQUET:
        safe_to_parquet(artifacts.predictions, out_dir / "predictions_full.parquet")
    else:
        predictions_sample.to_csv(out_dir / "predictions_sample.csv", index=False)

    write_json(out_dir / "metadata.json", artifacts.metadata)
    write_json(model_done_flag(out_dir), {
        "model_name": artifacts.model_name,
        "fold_name": artifacts.fold_name,
        "partition_name": artifacts.partition_name,
        "excel_path": str(excel_path),
        "summary_csv_path": str(out_dir / "summary_overall.csv"),
    })

def load_model_artifacts_if_done(spec: FoldSpec, model_name: str) -> Optional[ModelArtifacts]:
    out_dir = model_output_dir(spec, model_name)
    done = model_done_flag(out_dir)
    if not done.exists():
        return None

    try:
        summary_df = pd.read_csv(out_dir / "summary_overall.csv")
        regression_by_horizon = pd.read_csv(out_dir / "regression_by_horizon.csv")
        positive_by_horizon = pd.read_csv(out_dir / "positive_by_horizon.csv")
        outbreak_by_horizon = pd.read_csv(out_dir / "outbreak_by_horizon.csv")
        outbreak_confusion_by_horizon = pd.read_csv(out_dir / "outbreak_confusion_by_horizon.csv") if (out_dir / "outbreak_confusion_by_horizon.csv").exists() else pd.DataFrame()
        anticipation_summary = pd.read_csv(out_dir / "anticipation_summary.csv")
        anticipation_event_level = pd.read_csv(out_dir / "anticipation_event_level.csv")
        error_strata = pd.read_csv(out_dir / "error_strata.csv")
        training_history = pd.read_csv(out_dir / "training_history.csv") if (out_dir / "training_history.csv").exists() else pd.DataFrame()
        top_checkpoints = pd.read_csv(out_dir / "top_checkpoints.csv") if (out_dir / "top_checkpoints.csv").exists() else pd.DataFrame()
        metrics_by_municipality = pd.read_csv(out_dir / "metrics_by_municipality.csv") if (out_dir / "metrics_by_municipality.csv").exists() else None
        metadata = read_json(out_dir / "metadata.json", default={})

        pred_path_parquet = out_dir / "predictions_full.parquet"
        pred_path_csv = out_dir / "predictions_sample.csv"
        predictions = safe_read_dataframe(pred_path_parquet) if pred_path_parquet.exists() else safe_read_dataframe(pred_path_csv)
        if predictions is None:
            predictions = pd.DataFrame()

        overall = dict(summary_df.iloc[0].to_dict())

        return ModelArtifacts(
            model_name=model_name,
            fold_name=spec.fold_name,
            partition_name=spec.partition_name,
            overall=overall,
            regression_by_horizon=regression_by_horizon,
            positive_by_horizon=positive_by_horizon,
            outbreak_by_horizon=outbreak_by_horizon,
            outbreak_confusion_by_horizon=outbreak_confusion_by_horizon,
            anticipation_summary=anticipation_summary,
            anticipation_event_level=anticipation_event_level,
            error_strata=error_strata,
            predictions=predictions,
            training_history=training_history,
            top_checkpoints=top_checkpoints,
            metrics_by_municipality=metrics_by_municipality,
            metadata=metadata,
        )
    except Exception as e:
        print(f"[{spec.fold_name}][{model_name}] Error leyendo artefactos previos; se reentrenará. Detalle: {e}")
        return None

# =========================
# 20. MODEL DISPATCHER
# =========================
def run_model(spec: FoldSpec, model_name: str, refit_epochs: Optional[int] = None) -> ModelArtifacts:
    if model_name == "NAIVE":
        return run_naive_model(spec)
    if model_name == "ARIMA":
        return run_arima_model(spec)
    if model_name == "LSTM":
        return run_lstm_model(spec, refit_epochs=refit_epochs)
    if model_name == "TFT":
        return run_tft_model(spec, refit_epochs=refit_epochs)
    raise ValueError(f"Modelo no soportado: {model_name}")

# =========================
# 21. FOLD-LEVEL COMPARISON REPORT
# =========================
def _newey_west_long_run_variance(x: np.ndarray, max_lag: int) -> float:
    """
    Estimador HAC tipo Newey-West para la varianza de largo plazo
    de una serie univariada x.
    """
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    n = len(x)
    if n <= 1:
        return np.nan

    x_centered = x - np.mean(x)
    gamma0 = np.dot(x_centered, x_centered) / n
    lrv = gamma0

    for lag in range(1, max_lag + 1):
        w = 1.0 - lag / (max_lag + 1.0)
        gamma = np.dot(x_centered[lag:], x_centered[:-lag]) / n
        lrv += 2.0 * w * gamma

    return float(lrv)


def diebold_mariano_hln_test(
    y_true: np.ndarray,
    y_pred_a: np.ndarray,
    y_pred_b: np.ndarray,
    h: int,
    loss: str = "SE",
    model_a_name: str = "A",
    model_b_name: str = "B",
) -> Dict[str, Any]:
    """
    Diebold-Mariano con corrección de Harvey-Leybourne-Newbold (HLN).
    Compara precisión predictiva entre dos modelos usando la serie
    de diferenciales de pérdida.

    loss:
      - "SE": squared error
      - "AE": absolute error

    Convención:
      d_t = L_a - L_b
      mean(d) > 0  => B mejor (menor pérdida promedio)
      mean(d) < 0  => A mejor
    """
    y_true = np.asarray(y_true, dtype=float)
    y_pred_a = np.asarray(y_pred_a, dtype=float)
    y_pred_b = np.asarray(y_pred_b, dtype=float)

    mask = np.isfinite(y_true) & np.isfinite(y_pred_a) & np.isfinite(y_pred_b)
    y_true = y_true[mask]
    y_pred_a = y_pred_a[mask]
    y_pred_b = y_pred_b[mask]

    n = len(y_true)
    if n < max(10, h + 2):
        return {
            "n_obs": int(n),
            "loss": loss,
            "dm_stat": np.nan,
            "p_value_two_sided": np.nan,
            "mean_loss_diff_a_minus_b": np.nan,
            "better_model_by_mean_loss": None,
            "model_a": model_a_name,
            "model_b": model_b_name,
        }

    err_a = y_true - y_pred_a
    err_b = y_true - y_pred_b

    if loss.upper() == "SE":
        d = (err_a ** 2) - (err_b ** 2)
    elif loss.upper() == "AE":
        d = np.abs(err_a) - np.abs(err_b)
    else:
        raise ValueError(f"loss no soportada: {loss}")

    d_bar = float(np.mean(d))
    lrv = _newey_west_long_run_variance(d, max_lag=max(h - 1, 0))

    if (not np.isfinite(lrv)) or (lrv <= 0):
        return {
            "n_obs": int(n),
            "loss": loss,
            "dm_stat": np.nan,
            "p_value_two_sided": np.nan,
            "mean_loss_diff_a_minus_b": d_bar,
            "better_model_by_mean_loss": (
                model_b_name if d_bar > 0 else model_a_name if d_bar < 0 else "tie"
            ),
            "model_a": model_a_name,
            "model_b": model_b_name,
        }

    dm_stat = d_bar / np.sqrt(lrv / n)

    # Corrección de Harvey-Leybourne-Newbold para muestras finitas y horizonte h
    hln_factor = np.sqrt((n + 1 - 2 * h + (h * (h - 1) / n)) / n)
    dm_hln = float(dm_stat * hln_factor)

    # Aproximación t con n-1 grados de libertad
    p_value = float(2.0 * stats.t.sf(np.abs(dm_hln), df=n - 1))

    return {
        "n_obs": int(n),
        "loss": loss,
        "dm_stat": dm_hln,
        "p_value_two_sided": p_value,
        "mean_loss_diff_a_minus_b": d_bar,
        "better_model_by_mean_loss": (
            model_b_name if d_bar > 0 else model_a_name if d_bar < 0 else "tie"
        ),
        "model_a": model_a_name,
        "model_b": model_b_name,
    }


def mcnemar_paired_test_from_binary_predictions(
    y_true_bin: np.ndarray,
    y_pred_a_bin: np.ndarray,
    y_pred_b_bin: np.ndarray,
    model_a_name: str = "A",
    model_b_name: str = "B",
    exact: bool = True,
) -> Dict[str, Any]:
    """
    McNemar sobre corrección/incorrectitud emparejada de dos clasificadores
    evaluados sobre las mismas observaciones.
    """
    y_true_bin = np.asarray(y_true_bin, dtype=int)
    y_pred_a_bin = np.asarray(y_pred_a_bin, dtype=int)
    y_pred_b_bin = np.asarray(y_pred_b_bin, dtype=int)

    mask = np.isfinite(y_true_bin) & np.isfinite(y_pred_a_bin) & np.isfinite(y_pred_b_bin)
    y_true_bin = y_true_bin[mask]
    y_pred_a_bin = y_pred_a_bin[mask]
    y_pred_b_bin = y_pred_b_bin[mask]

    correct_a = (y_pred_a_bin == y_true_bin).astype(int)
    correct_b = (y_pred_b_bin == y_true_bin).astype(int)

    both_correct = int(np.sum((correct_a == 1) & (correct_b == 1)))
    a_correct_b_wrong = int(np.sum((correct_a == 1) & (correct_b == 0)))
    a_wrong_b_correct = int(np.sum((correct_a == 0) & (correct_b == 1)))
    both_wrong = int(np.sum((correct_a == 0) & (correct_b == 0)))

    table = np.array([
        [both_correct, a_correct_b_wrong],
        [a_wrong_b_correct, both_wrong]
    ])

    res = mcnemar(table, exact=exact, correction=not exact)

    acc_a = float(np.mean(correct_a))
    acc_b = float(np.mean(correct_b))

    return {
        "n_obs": int(len(y_true_bin)),
        "model_a": model_a_name,
        "model_b": model_b_name,
        "accuracy_a": acc_a,
        "accuracy_b": acc_b,
        "accuracy_diff_b_minus_a": acc_b - acc_a,
        "both_correct": both_correct,
        "a_correct_b_wrong": a_correct_b_wrong,
        "a_wrong_b_correct": a_wrong_b_correct,
        "both_wrong": both_wrong,
        "mcnemar_stat": float(res.statistic) if res.statistic is not None else np.nan,
        "p_value_two_sided": float(res.pvalue) if res.pvalue is not None else np.nan,
        "better_model_by_accuracy": (
            model_b_name if acc_b > acc_a else model_a_name if acc_a > acc_b else "tie"
        ),
        "exact_test": bool(exact),
    }


def build_final_test_significance_tables(
    spec: FoldSpec,
    artifacts_dict: Dict[str, ModelArtifacts],
    reference_model: str = "TFT",
) -> Dict[str, pd.DataFrame]:
    """
    Para final_test:
      - DM por horizonte para regresión (SE y AE), TFT vs cada competidor.
      - McNemar por horizonte para clasificación binaria:
          * positivos
          * outbreaks
    """
    if spec.partition_name != "final_test":
        return {}

    key_cols = [GROUP_COL, "prediction_start_time_idx", "horizon", "target_time_idx"]

    if reference_model not in artifacts_dict:
        return {}

    ref_pred = artifacts_dict[reference_model].predictions.copy()
    ref_pred = ref_pred[key_cols + [
        "y_true", "y_pred",
        "is_positive_true", "is_positive_pred_model",
        "is_outbreak_true", "is_outbreak_pred_model"
    ]].rename(columns={
        "y_pred": f"y_pred_{reference_model}",
        "is_positive_pred_model": f"is_positive_pred_{reference_model}",
        "is_outbreak_pred_model": f"is_outbreak_pred_{reference_model}",
    })

    dm_rows = []
    positive_rows = []
    outbreak_rows = []

    for model_name, art in artifacts_dict.items():
        if model_name == reference_model:
            continue

        cmp_pred = art.predictions.copy()
        cmp_pred = cmp_pred[key_cols + [
            "y_pred",
            "is_positive_pred_model",
            "is_outbreak_pred_model"
        ]].rename(columns={
            "y_pred": f"y_pred_{model_name}",
            "is_positive_pred_model": f"is_positive_pred_{model_name}",
            "is_outbreak_pred_model": f"is_outbreak_pred_{model_name}",
        })

        merged = ref_pred.merge(cmp_pred, on=key_cols, how="inner")

        for h in sorted(merged["horizon"].unique()):
            g = merged.loc[merged["horizon"] == h].copy()

            # DM para regresión
            dm_se = diebold_mariano_hln_test(
                y_true=g["y_true"].values,
                y_pred_a=g[f"y_pred_{reference_model}"].values,
                y_pred_b=g[f"y_pred_{model_name}"].values,
                h=int(h),
                loss="SE",
                model_a_name=reference_model,
                model_b_name=model_name,
            )
            dm_ae = diebold_mariano_hln_test(
                y_true=g["y_true"].values,
                y_pred_a=g[f"y_pred_{reference_model}"].values,
                y_pred_b=g[f"y_pred_{model_name}"].values,
                h=int(h),
                loss="AE",
                model_a_name=reference_model,
                model_b_name=model_name,
            )

            dm_rows.append({
                "reference_model": reference_model,
                "compared_model": model_name,
                "horizon": int(h),
                "n_obs": int(len(g)),
                "loss": "SE",
                **{k: v for k, v in dm_se.items() if k not in ["model_a", "model_b", "loss", "n_obs"]},
            })
            dm_rows.append({
                "reference_model": reference_model,
                "compared_model": model_name,
                "horizon": int(h),
                "n_obs": int(len(g)),
                "loss": "AE",
                **{k: v for k, v in dm_ae.items() if k not in ["model_a", "model_b", "loss", "n_obs"]},
            })

            # McNemar para positivos
            mc_pos = mcnemar_paired_test_from_binary_predictions(
                y_true_bin=g["is_positive_true"].values,
                y_pred_a_bin=g[f"is_positive_pred_{reference_model}"].values,
                y_pred_b_bin=g[f"is_positive_pred_{model_name}"].values,
                model_a_name=reference_model,
                model_b_name=model_name,
                exact=True,
            )
            positive_rows.append({
                "reference_model": reference_model,
                "compared_model": model_name,
                "horizon": int(h),
                **{k: v for k, v in mc_pos.items() if k not in ["model_a", "model_b"]},
            })

            # McNemar para outbreaks
            mc_out = mcnemar_paired_test_from_binary_predictions(
                y_true_bin=g["is_outbreak_true"].values,
                y_pred_a_bin=g[f"is_outbreak_pred_{reference_model}"].values,
                y_pred_b_bin=g[f"is_outbreak_pred_{model_name}"].values,
                model_a_name=reference_model,
                model_b_name=model_name,
                exact=True,
            )
            outbreak_rows.append({
                "reference_model": reference_model,
                "compared_model": model_name,
                "horizon": int(h),
                **{k: v for k, v in mc_out.items() if k not in ["model_a", "model_b"]},
            })

    return {
        "dm_regression_tests": pd.DataFrame(dm_rows),
        "mcnemar_positive_tests": pd.DataFrame(positive_rows),
        "mcnemar_outbreak_tests": pd.DataFrame(outbreak_rows),
    }

def build_fold_comparison_report(spec: FoldSpec, artifacts_dict: Dict[str, ModelArtifacts]):
    fold_dir = spec.output_dir
    ensure_dir(fold_dir)

    summary_rows = []
    regression_rows = []
    positive_rows = []
    outbreak_rows = []
    outbreak_confusion_rows = []
    anticipation_rows = []
    error_rows = []

    for model_name in RUN_MODELS:
        art = artifacts_dict.get(model_name, None)
        if art is None:
            continue
        summary_rows.append(pd.DataFrame([art.overall]))
        regression_rows.append(art.regression_by_horizon)
        positive_rows.append(art.positive_by_horizon)
        outbreak_rows.append(art.outbreak_by_horizon)
        outbreak_confusion_rows.append(art.outbreak_confusion_by_horizon)

        ant = art.anticipation_summary.copy()
        ant["model_name"] = model_name
        ant["fold_name"] = spec.fold_name
        ant["partition_name"] = spec.partition_name
        anticipation_rows.append(ant)

        error_rows.append(art.error_strata)

    summary_df = reorder_columns(pd.concat(summary_rows, ignore_index=True), SUMMARY_COLUMNS_ORDER) if summary_rows else pd.DataFrame()
    regression_df = reorder_columns(pd.concat(regression_rows, ignore_index=True), REGRESSION_COLUMNS_ORDER) if regression_rows else pd.DataFrame()
    positive_df = reorder_columns(pd.concat(positive_rows, ignore_index=True), POSITIVE_COLUMNS_ORDER) if positive_rows else pd.DataFrame()
    outbreak_df = reorder_columns(pd.concat(outbreak_rows, ignore_index=True), OUTBREAK_COLUMNS_ORDER) if outbreak_rows else pd.DataFrame()
    outbreak_confusion_df = pd.concat(outbreak_confusion_rows, ignore_index=True) if outbreak_confusion_rows else pd.DataFrame()
    anticipation_df = pd.concat(anticipation_rows, ignore_index=True) if anticipation_rows else pd.DataFrame()
    error_df = pd.concat(error_rows, ignore_index=True) if error_rows else pd.DataFrame()

    key_match_df = validate_prediction_keys_across_models(artifacts_dict, strict=True)

    rank_cols = [
        "model_name", "MAE", "RMSE", "positive_model_f1",
        "outbreak_model_f1", "outbreak_model_f1_relaxed",
        "skill_MAE_vs_naive", "outbreak_f1_gain_vs_naive"
    ]
    ranking_df = summary_df[[c for c in rank_cols if c in summary_df.columns]].copy() if len(summary_df) > 0 else pd.DataFrame()

    sheets = {
        "summary_overall": summary_df,
        "regression_by_horizon": regression_df,
        "positive_by_horizon": positive_df,
        "outbreak_by_horizon": outbreak_df,
        "outbreak_confusion_by_horizon": outbreak_confusion_df,
        "anticipation_summary": anticipation_df,
        "error_strata": error_df,
        "model_ranking": ranking_df,
        "prediction_key_match": key_match_df,
    }

    if spec.partition_name == "final_test":
        sig_tables = build_final_test_significance_tables(spec, artifacts_dict, reference_model="TFT")
        for k, v in sig_tables.items():
            sheets[k] = v

    excel_path = fold_dir / f"{spec.fold_name}_comparison_report.xlsx"
    write_excel_report(excel_path, sheets)

    summary_df.to_csv(fold_dir / "comparison_summary_overall.csv", index=False)
    regression_df.to_csv(fold_dir / "comparison_regression_by_horizon.csv", index=False)
    positive_df.to_csv(fold_dir / "comparison_positive_by_horizon.csv", index=False)
    outbreak_df.to_csv(fold_dir / "comparison_outbreak_by_horizon.csv", index=False)
    outbreak_confusion_df.to_csv(fold_dir / "comparison_outbreak_confusion_by_horizon.csv", index=False)
    key_match_df.to_csv(fold_dir / "comparison_prediction_key_match.csv", index=False)

    if spec.partition_name == "final_test":
        sig_tables = build_final_test_significance_tables(spec, artifacts_dict, reference_model="TFT")
        for k, v in sig_tables.items():
            v.to_csv(fold_dir / f"{k}.csv", index=False)

    write_json(fold_dir / "fold_comparison_done.json", {
        "fold_name": spec.fold_name,
        "partition_name": spec.partition_name,
        "excel_path": str(excel_path),
        "models_included": list(artifacts_dict.keys()),
    })
# =========================
# 22. CV / FINAL SUMMARY UPDATE
# =========================
def append_or_replace_summary_row(csv_path: Path, new_row: Dict[str, Any], key_cols: List[str]):
    ensure_dir(csv_path.parent)
    new_df = pd.DataFrame([new_row])
    if csv_path.exists():
        old = pd.read_csv(csv_path)
        if len(old) > 0:
            mask = np.ones(len(old), dtype=bool)
            for c in key_cols:
                mask &= old[c].astype(str) == str(new_row[c])
            old = old.loc[~mask].copy()
            out = pd.concat([old, new_df], ignore_index=True)
        else:
            out = new_df
    else:
        out = new_df
    out.to_csv(csv_path, index=False)

def refresh_global_summary(root_dir: Path, partition_name: str):
    model_summary_rows = []

    for fold_dir in sorted(root_dir.glob("*")):
        if not fold_dir.is_dir():
            continue
        comparison_csv = fold_dir / "comparison_summary_overall.csv"
        if comparison_csv.exists():
            try:
                df_cmp = pd.read_csv(comparison_csv)
                model_summary_rows.append(df_cmp)
            except Exception:
                pass

    if len(model_summary_rows) == 0:
        return

    summary_all = pd.concat(model_summary_rows, ignore_index=True)
    summary_all = reorder_columns(summary_all, SUMMARY_COLUMNS_ORDER)
    mean_metrics = summary_all.groupby("model_name").mean(numeric_only=True).reset_index()

    excel_path = root_dir / f"{partition_name}_summary_report.xlsx"
    write_excel_report(excel_path, {
        "summary_overall": summary_all,
        "model_mean_metrics": mean_metrics,
    })

    summary_all.to_csv(root_dir / f"{partition_name}_summary_overall.csv", index=False)
    mean_metrics.to_csv(root_dir / f"{partition_name}_model_mean_metrics.csv", index=False)

# =========================
# 23. REFIT EPOCH SELECTION
# =========================
def selected_epoch_from_artifact(art: Optional[ModelArtifacts], default_epochs: int) -> int:
    if art is None:
        return default_epochs
    v = art.overall.get("selected_checkpoint_epoch_1based", np.nan)
    if pd.isna(v):
        return default_epochs
    return int(v)

def median_selected_epochs_from_cv(cv_root: Path, model_name: str, default_epochs: int) -> int:
    vals = []
    for fold_dir in sorted(cv_root.glob("fold_val_*")):
        # Primero intentamos el JSON ligero generado por la CV reducida
        light_flag = fold_dir / model_name.lower() / "_CV_LIGHT_DONE.json"
        if light_flag.exists():
            info = read_json(light_flag, default={})
            v = info.get("selected_checkpoint_epoch_1based", None)
            if v is not None and not pd.isna(v):
                vals.append(int(v))
                continue

        # Fallback al summary completo si existe (modo legado)
        summary_csv = fold_dir / model_name.lower() / "summary_overall.csv"
        if summary_csv.exists():
            try:
                df_s = pd.read_csv(summary_csv)
                if "selected_checkpoint_epoch_1based" in df_s.columns:
                    v = df_s["selected_checkpoint_epoch_1based"].iloc[0]
                    if not pd.isna(v):
                        vals.append(int(v))
            except Exception:
                pass

    if len(vals) == 0:
        return default_epochs
    return int(np.median(vals))

# =========================
# 24. PRE-RUN TECHNICAL NOTE SAVER
# =========================
def save_run_config(df_model: pd.DataFrame):
    run_config = {
        "SEED": SEED,
        "RESULTS_DIR": str(RESULTS_DIR),
        "RUN_MODELS": RUN_MODELS,
        "MAX_ENCODER_LENGTH": MAX_ENCODER_LENGTH,
        "MIN_ENCODER_LENGTH": MIN_ENCODER_LENGTH,
        "MAX_PREDICTION_LENGTH": MAX_PREDICTION_LENGTH,
        "VALIDATION_YEARS": VALIDATION_YEARS,
        "TEST_YEAR": TEST_YEAR,
        "POSITIVE_CLASSIFICATION_THRESHOLD": POSITIVE_CLASSIFICATION_THRESHOLD,
        "OUTBREAK_BASELINE_WINDOW": OUTBREAK_BASELINE_WINDOW,
        "OUTBREAK_BASELINE_MIN_HISTORY": OUTBREAK_BASELINE_MIN_HISTORY,
        "OUTBREAK_GROWTH_RATE": OUTBREAK_GROWTH_RATE,
        "OUTBREAK_MIN_CASES_ABS": OUTBREAK_MIN_CASES_ABS,
        "OUTBREAK_BASELINE_FLOOR": OUTBREAK_BASELINE_FLOOR,
        "WEIGHT_SCHEME": "dynamic_outbreak_single_class",
        "LOSS_W_ZERO": LOSS_W_ZERO,
        "LOSS_W_POSITIVE_NON_OUTBREAK": LOSS_W_POSITIVE_NON_OUTBREAK,
        "LOSS_W_OUTBREAK": LOSS_W_OUTBREAK,
        "USE_ACTIVE_MUNICIPALITIES": USE_ACTIVE_MUNICIPALITIES,
        "FUTURE_CLIMATE_IS_TRULY_KNOWN": FUTURE_CLIMATE_IS_TRULY_KNOWN,
        "TFT": {
            "batch_size": TFT_BATCH_SIZE,
            "max_epochs": TFT_MAX_EPOCHS,
            "learning_rate": TFT_LEARNING_RATE,
            "hidden_size": TFT_HIDDEN_SIZE,
            "attention_head_size": TFT_ATTENTION_HEAD_SIZE,
            "dropout": TFT_DROPOUT,
            "hidden_continuous_size": TFT_HIDDEN_CONT_SIZE,
        },
        "LSTM": {
            "batch_size": LSTM_BATCH_SIZE,
            "max_epochs": LSTM_MAX_EPOCHS,
            "learning_rate": LSTM_LEARNING_RATE,
            "hidden_size": LSTM_HIDDEN_SIZE,
            "num_layers": LSTM_NUM_LAYERS,
            "dropout": LSTM_DROPOUT,
            "embed_dim": LSTM_EMBED_DIM,
        },
        "ARIMA": {
            "use_auto_arima": bool(ARIMA_USE_AUTO_ARIMA and PMDARIMA_AVAILABLE),
            "max_history": ARIMA_MAX_HISTORY,
            "seasonal": False,
            "auto_arima_search": {
                "start_p": 0,
                "start_q": 0,
                "max_p": 4,
                "max_q": 4,
                "max_d": 1,
                "stepwise": True,
                "n_fits": 20,
                "with_intercept": True,
            },
        },
        "FINAL_TEST_SIGNIFICANCE": {
            "reference_model": "TFT",
            "regression_test": "Diebold-Mariano_HLN",
            "classification_test_positive": "McNemar_exact",
            "classification_test_outbreak": "McNemar_exact",
        },
        "n_rows": int(len(df_model)),
        "n_municipalities": int(df_model[GROUP_COL].nunique()),
        "years": sorted(df_model[YEAR_COL].unique().tolist()),
    }
    write_json(RESULTS_DIR / "run_config.json", run_config)

# =========================
# 25. SINGLE FOLD EXECUTION
# =========================
def run_single_fold_cv_light(spec: FoldSpec):
    """
    Ejecución de fold de validación cruzada en modo ligero.

    Durante la CV solo necesitamos dos cosas para alimentar el test final:
    el val_loss del mejor checkpoint y el número de época seleccionada.
    Esto evita generar reportes Excel/CSV completos por fold y reduce
    drásticamente el tiempo de cómputo y el uso de disco.

    Solo se entrena TFT, que es el modelo cuyo número óptimo de épocas
    queremos transferir al test final. NAIVE, ARIMA y LSTM se reservan
    al test final: NAIVE y ARIMA porque son baselines deterministas, y
    LSTM porque empíricamente entrena bien con un número fijo de épocas
    (3) y reentrenarlo en cada fold dispararía el tiempo de cómputo sin
    aportar información útil.
    """
    print("\n" + "=" * 100)
    print(f"CV LIGERA | {spec.fold_name}")
    print("=" * 100)

    cv_models = [m for m in RUN_MODELS if m == "TFT"]

    for model_name in cv_models:
        out_dir = model_output_dir(spec, model_name)
        light_flag = out_dir / "_CV_LIGHT_DONE.json"

        if ALLOW_RESUME and light_flag.exists() and not FORCE_RETRAIN_COMPLETED_MODELS:
            print(f"[{spec.fold_name}][{model_name}] CV ligera ya completada. Se reutiliza.")
            continue

        print(f"\n[{spec.fold_name}] Entrenando {model_name} (modo ligero)...")
        art = run_model(spec, model_name, refit_epochs=None)

        selected_epoch = int(art.overall.get("selected_checkpoint_epoch_1based", 0) or 0)
        selected_path = art.overall.get("selected_checkpoint_path", "")

        # Recuperamos el mejor val_loss desde el historial de entrenamiento
        val_loss_best = np.nan
        if art.training_history is not None and "val_loss" in art.training_history.columns:
            try:
                val_loss_best = float(art.training_history["val_loss"].dropna().min())
            except Exception:
                val_loss_best = np.nan

        write_json(light_flag, {
            "model_name": model_name,
            "fold_name": spec.fold_name,
            "selected_checkpoint_epoch_1based": selected_epoch,
            "selected_checkpoint_path": str(selected_path),
            "best_val_loss": val_loss_best,
        })

        # Guardamos también summary mínimo para que median_selected_epochs_from_cv
        # siga funcionando sin cambios.
        summary_min = pd.DataFrame([{
            "model_name": model_name,
            "fold_name": spec.fold_name,
            "partition_name": spec.partition_name,
            "selected_checkpoint_epoch_1based": selected_epoch,
            "best_val_loss": val_loss_best,
        }])
        summary_min.to_csv(out_dir / "summary_overall.csv", index=False)
        write_json(model_done_flag(out_dir), {
            "model_name": model_name,
            "fold_name": spec.fold_name,
            "partition_name": spec.partition_name,
            "mode": "cv_light",
        })

        del art
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    print(f"[{spec.fold_name}] CV ligera completada.")


def _score_tft_trial(art: ModelArtifacts) -> float:
    """
    Score escalar para comparar configuraciones durante el tuning.

    Pondera Outbreak F1, MAE sobre positivos y MAE sobre outbreaks contra
    falsas alarmas y sesgo al alza en la clase cero. El término de sesgo al
    alza se mide como la predicción media sobre observaciones donde la verdad
    es cero: configuraciones que inflan esa zona son penalizadas porque
    deterioran el MAE global sin aportar detección de brotes.

    Mayor es mejor.
    """
    o = art.overall
    out_f1 = float(o.get("outbreak_model_f1", 0.0) or 0.0)
    out_far = float(o.get("outbreak_model_false_alarm_rate", 0.0) or 0.0)
    pos_f1 = float(o.get("positive_model_f1", 0.0) or 0.0)
    mae_out = float(o.get("MAE_on_true_outbreaks", 0.0) or 0.0)

    mae_pos = np.nan
    rh = art.regression_by_horizon
    if rh is not None and "MAE_nonzero" in rh.columns:
        mae_pos = float(rh["MAE_nonzero"].mean())
    if not np.isfinite(mae_pos):
        mae_pos = float(o.get("MAE", 0.0) or 0.0)

    mae_pos_norm = mae_pos / (mae_pos + 6.0)
    mae_out_norm = mae_out / (mae_out + 6.0)

    mean_pred_on_zero = float(o.get("mean_pred_when_true_zero", 0.0) or 0.0)
    bias_zero_norm = mean_pred_on_zero / (mean_pred_on_zero + 2.0)

    return (
        1.5 * out_f1
        + 0.4 * pos_f1
        - 0.6 * out_far
        - 0.5 * mae_pos_norm
        - 0.4 * mae_out_norm
        - 0.4 * bias_zero_norm
    )


def _build_tuning_subsample(df_model: pd.DataFrame, frac: float, seed: int = SEED) -> pd.DataFrame:
    """
    Submuestreo estratificado de municipios para acelerar el tuning.

    Los municipios se agrupan en cuatro cuartiles por volumen histórico total
    de casos y de cada cuartil se toma una fracción uniforme. Esto preserva la
    mezcla representativa (municipios muy activos, intermedios, poco activos)
    y reduce el número de filas por trial sin sesgar el ranking entre
    configuraciones de pesos.
    """
    muni_totals = (
        df_model.groupby(GROUP_COL)[TARGET_COL].sum().reset_index(name="total")
    )
    muni_totals["q"] = pd.qcut(
        muni_totals["total"].rank(method="first"),
        q=4, labels=False, duplicates="drop",
    )
    rng = np.random.default_rng(seed)
    keep = []
    for _, grp in muni_totals.groupby("q"):
        n_keep = max(1, int(round(len(grp) * frac)))
        idx = rng.choice(len(grp), size=n_keep, replace=False)
        keep.extend(grp.iloc[idx][GROUP_COL].astype(str).tolist())
    sub = df_model[df_model[GROUP_COL].astype(str).isin(set(keep))].copy()
    print(
        f"[tuning subsample] municipios: {len(keep)} / {muni_totals.shape[0]} "
        f"({100.0 * len(keep)/muni_totals.shape[0]:.1f}%) | filas: {len(sub):,}"
    )
    return sub


def tune_tft_hyperparameters(df_model: pd.DataFrame, cv_root: Path) -> Dict[str, Any]:
    """
    Búsqueda de pesos de pérdida para TFT con arquitectura fija.

    La arquitectura del TFT (hidden_size, attention_head_size,
    hidden_continuous_size) se fija a la especificación final reportada en
    la tesis. El tuning se concentra exclusivamente en la búsqueda de la
    mejor combinación de pesos (w_zero, w_positive, w_outbreak) sobre el
    fold de validación TUNE_TFT_FOLD_YEAR.

    Cada trial entrena TUNE_TFT_MAX_EPOCHS épocas sobre un submuestreo
    estratificado del dataset. El resultado de cada trial se persiste en
    un JSON, lo que permite reanudar la búsqueda si el proceso se
    interrumpe: los trials ya terminados se reutilizan sin reentrenar.

    El año TEST_YEAR (2024) permanece intocado durante todo el procedimiento.
    """
    global TFT_HIDDEN_SIZE, TFT_ATTENTION_HEAD_SIZE, TFT_HIDDEN_CONT_SIZE
    global LOSS_W_ZERO, LOSS_W_POSITIVE_NON_OUTBREAK, LOSS_W_OUTBREAK

    print("\n" + "=" * 100)
    print(
        f"TUNING TFT | fold = {TUNE_TFT_FOLD_YEAR} | epochs = {TUNE_TFT_MAX_EPOCHS} "
        f"| subsample = {TUNE_SUBSAMPLE_FRACTION:.0%}"
    )
    print("=" * 100)

    tune_dir = cv_root / f"_tuning_tft_{TUNE_TFT_FOLD_YEAR}"
    ensure_dir(tune_dir)

    df_tune = _build_tuning_subsample(df_model, frac=TUNE_SUBSAMPLE_FRACTION, seed=SEED)

    original_max_epochs = TFT_MAX_EPOCHS
    all_results = []

    def _run_trial(trial_name: str, cfg: Dict[str, Any]) -> Tuple[float, Dict[str, Any]]:
        trial_dir = tune_dir / trial_name
        ensure_dir(trial_dir)

        # Si ya existe un resultado serializado de este trial, se reutiliza.
        trial_result_path = trial_dir / "_TRIAL_DONE.json"
        if trial_result_path.exists():
            saved = read_json(trial_result_path, default={})
            if saved and "score" in saved:
                print(f"   [reutilizado] score={saved['score']:.4f}")
                return float(saved["score"]), saved

        # Limpieza preventiva de checkpoints heredados.
        shutil.rmtree(trial_dir / "tft", ignore_errors=True)

        globals()["TFT_HIDDEN_SIZE"] = cfg["hidden_size"]
        globals()["TFT_ATTENTION_HEAD_SIZE"] = cfg["attention_head_size"]
        globals()["TFT_HIDDEN_CONT_SIZE"] = cfg["hidden_continuous_size"]
        globals()["LOSS_W_ZERO"] = cfg["w_zero"]
        globals()["LOSS_W_POSITIVE_NON_OUTBREAK"] = cfg["w_positive"]
        globals()["LOSS_W_OUTBREAK"] = cfg["w_outbreak"]

        spec = build_fold_spec(
            df_model=df_tune,
            fold_name=trial_name,
            eval_start_year=TUNE_TFT_FOLD_YEAR,
            partition_name="cv",
            output_dir=trial_dir,
        )

        try:
            globals()["TFT_MAX_EPOCHS"] = TUNE_TFT_MAX_EPOCHS
            art = run_tft_model(spec, refit_epochs=TUNE_TFT_MAX_EPOCHS)
        finally:
            globals()["TFT_MAX_EPOCHS"] = original_max_epochs

        score = _score_tft_trial(art)
        row = {
            "trial": trial_name,
            **cfg,
            "score": score,
            "outbreak_f1": float(art.overall.get("outbreak_model_f1", np.nan)),
            "outbreak_far": float(art.overall.get("outbreak_model_false_alarm_rate", np.nan)),
            "positive_f1": float(art.overall.get("positive_model_f1", np.nan)),
            "MAE": float(art.overall.get("MAE", np.nan)),
            "MAE_on_true_outbreaks": float(art.overall.get("MAE_on_true_outbreaks", np.nan)),
            "mean_pred_when_true_zero": float(art.overall.get("mean_pred_when_true_zero", np.nan)),
        }
        print(
            f"   -> score={score:.4f} | out_f1={row['outbreak_f1']:.3f} "
            f"| far={row['outbreak_far']:.3f} | mae={row['MAE']:.3f} "
            f"| mae_out={row['MAE_on_true_outbreaks']:.3f} "
            f"| bias_zero={row['mean_pred_when_true_zero']:.3f}"
        )

        write_json(trial_result_path, row)
        shutil.rmtree(trial_dir / "tft" / "_temp_ckpt", ignore_errors=True)
        del art
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        return score, row

    # Arquitectura fija reportada en la tesis (la misma que usa la CV).
    final_arch_cfg = {
        "hidden_size": TFT_HIDDEN_SIZE,
        "attention_head_size": TFT_ATTENTION_HEAD_SIZE,
        "hidden_continuous_size": TFT_HIDDEN_CONT_SIZE,
    }
    print(f"\nArquitectura fija: {final_arch_cfg}")

    # Grid de pesos: referencia conservadora, variantes con w_zero elevado
    # para combatir sesgo al alza sobre ceros y combinaciones con w_outbreak
    # mayor para presionar sensibilidad al brote.
    weight_grid = [
        {"w_zero": 1.0, "w_positive": 1.5, "w_outbreak": 6.25},
        {"w_zero": 1.5, "w_positive": 1.5, "w_outbreak": 6.25},
        {"w_zero": 1.5, "w_positive": 2.0, "w_outbreak": 7.0},
        {"w_zero": 2.0, "w_positive": 2.0, "w_outbreak": 7.5},
        {"w_zero": 1.0, "w_positive": 2.0, "w_outbreak": 8.5},
        {"w_zero": 1.5, "w_positive": 2.5, "w_outbreak": 9.0},
        {"w_zero": 2.0, "w_positive": 1.5, "w_outbreak": 6.5},
        {"w_zero": 1.25, "w_positive": 1.75, "w_outbreak": 7.0},
    ]

    print("\n--- Búsqueda de pesos de pérdida ---")
    best_score = -np.inf
    best_cfg = None
    for i, w in enumerate(weight_grid, start=1):
        cfg = {**final_arch_cfg, **w}
        print(f"\n[weights {i}/{len(weight_grid)}] {cfg}")
        score, row = _run_trial(f"weights_{i:02d}", cfg)
        all_results.append(row)
        if score > best_score:
            best_score = score
            best_cfg = cfg

    results_df = pd.DataFrame(all_results).sort_values("score", ascending=False)
    results_df.to_csv(tune_dir / "tuning_results.csv", index=False)
    write_json(tune_dir / "best_config.json", best_cfg or {})

    if best_cfg is not None:
        TFT_HIDDEN_SIZE = best_cfg["hidden_size"]
        TFT_ATTENTION_HEAD_SIZE = best_cfg["attention_head_size"]
        TFT_HIDDEN_CONT_SIZE = best_cfg["hidden_continuous_size"]
        LOSS_W_ZERO = best_cfg["w_zero"]
        LOSS_W_POSITIVE_NON_OUTBREAK = best_cfg["w_positive"]
        LOSS_W_OUTBREAK = best_cfg["w_outbreak"]
        print(f"\nMejor configuración: {best_cfg} | score={best_score:.4f}")

    return best_cfg or {}


def run_single_fold(spec: FoldSpec, cv_root: Optional[Path] = None):
    print("\n" + "=" * 100)
    print(f"INICIANDO {spec.fold_name} | partition={spec.partition_name}")
    print("=" * 100)

    artifacts_dict = {}

    for model_name in RUN_MODELS:
        print("\n" + "-" * 90)
        print(f"[{spec.fold_name}] Modelo: {model_name}")
        print("-" * 90)

        out_dir = model_output_dir(spec, model_name)

        if ALLOW_RESUME and (not FORCE_RETRAIN_COMPLETED_MODELS):
            loaded = load_model_artifacts_if_done(spec, model_name)
            if loaded is not None:
                print(f"[{spec.fold_name}][{model_name}] Ya completado previamente. Se reutilizan artefactos.")
                artifacts_dict[model_name] = loaded
                continue

        refit_epochs = None
        if spec.partition_name == "final_test":
            if model_name == "TFT" and cv_root is not None:
                refit_epochs = median_selected_epochs_from_cv(cv_root, "TFT", TFT_MAX_EPOCHS)
            if model_name == "LSTM":
                # La mediana óptima observada empíricamente fue 3 épocas.
                refit_epochs = 3

        art = run_model(spec, model_name, refit_epochs=refit_epochs)
        save_model_artifacts(spec, art)
        artifacts_dict[model_name] = art

        # Incremental fold comparison after each model finishes
        build_fold_comparison_report(spec, artifacts_dict)
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    if FORCE_REBUILD_FOLD_COMPARISON:
        build_fold_comparison_report(spec, artifacts_dict)

    print(f"[{spec.fold_name}] FOLD COMPLETADO.")
    return artifacts_dict


## Preparación de los datos

Aplica las mismas transformaciones que el pipeline original (validación
de columnas, cálculo de `time_idx`, codificación de semana epidemiológica,
etc.) y construye `spec_final`, el split de entrenamiento y evaluación
para 2024.


In [ ]:
print("Preparando datos base...")

required_cols = [GROUP_COL, TIME_COL, YEAR_COL, TARGET_COL, FLOW_COL] + CLIMATE_COLS + AGE_COLS
missing_cols = [c for c in required_cols if c not in df.columns]
if missing_cols:
    raise ValueError(f"Faltan columnas en df: {missing_cols}")

df = ensure_string_group(df.copy())
df_model = add_weekly_features(df)

print(f"Filas totales:        {len(df_model):,}")
print(f"Municipios:           {df_model[GROUP_COL].nunique():,}")
print(f"Años disponibles:     {sorted(df_model[YEAR_COL].unique())}")
print(f"% ceros global:       {100 * (df_model[TARGET_COL] == 0).mean():.2f}%")

# Construir el spec del test final (mismas transformaciones que el pipeline).
FINAL_ROOT = RESULTS_DIR / f"final_test_{TEST_YEAR}"
ensure_dir(FINAL_ROOT)

spec_final = build_fold_spec(
    df_model=df_model,
    fold_name=f"final_test_{TEST_YEAR}",
    eval_start_year=TEST_YEAR,
    partition_name="final_test",
    output_dir=FINAL_ROOT,
)

print(f"\nSpec construido: train={len(spec_final.train_df):,} filas, eval={len(spec_final.eval_df):,} filas")


## NAIVE

El modelo NAIVE no requiere entrenamiento y es determinista. Se recalcula
desde cero llamando a `run_naive_model`.


In [ ]:
art_naive = run_naive_model(spec_final)
save_model_artifacts(spec_final, art_naive)
print("\nNAIVE — métricas globales:")
print(f"  MAE:                {art_naive.overall.get('MAE', float('nan')):.4f}")
print(f"  RMSE:               {art_naive.overall.get('RMSE', float('nan')):.4f}")
print(f"  Outbreak F1:        {art_naive.overall.get('outbreak_model_f1', float('nan')):.4f}")
print(f"  Outbreak FAR:       {art_naive.overall.get('outbreak_model_false_alarm_rate', float('nan')):.4f}")


## ARIMA

Igual que NAIVE: ARIMA es determinista y se recalcula desde cero. Esta
celda puede demorar algunos minutos sobre el dataset completo porque
ajusta un modelo por municipio.


In [ ]:
art_arima = run_arima_model(spec_final)
save_model_artifacts(spec_final, art_arima)
print("\nARIMA — métricas globales:")
print(f"  MAE:                {art_arima.overall.get('MAE', float('nan')):.4f}")
print(f"  RMSE:               {art_arima.overall.get('RMSE', float('nan')):.4f}")
print(f"  Outbreak F1:        {art_arima.overall.get('outbreak_model_f1', float('nan')):.4f}")
print(f"  Outbreak FAR:       {art_arima.overall.get('outbreak_model_false_alarm_rate', float('nan')):.4f}")


## LSTM

Carga el checkpoint del LSTM y ejecuta predicción sobre el conjunto de
test. La inferencia es determinista (modelo en modo `eval`, sin dropout
y sin shuffle del DataLoader).


In [ ]:
import torch

train_loader, val_loader, group_to_idx, meta = build_lstm_dataloaders(spec_final)

# Reconstruir el modelo desde el checkpoint con los mismos hiperparámetros
# que se persistieron en hyper_parameters.
lstm_model = LSTMMultiHorizon.load_from_checkpoint(str(LSTM_CKPT), map_location="cpu")
lstm_model.eval()

# Inferencia sobre el conjunto de evaluación (val_loader del fold final).
pred_df_lstm_raw = collect_lstm_predictions(lstm_model, val_loader)
pred_df_lstm = prepare_predictions_for_evaluation(pred_df_lstm_raw, source_eval_df=spec_final.eval_df)

# Métricas
overall_lstm, reg_h_lstm, pos_h_lstm, out_h_lstm, conf_h_lstm, by_muni_lstm = compute_overall_and_horizon_metrics(
    pred_df=pred_df_lstm,
    model_name="LSTM",
    fold_name=spec_final.fold_name,
    partition_name=spec_final.partition_name,
)

event_lstm, anti_lstm = build_outbreak_anticipation_report(pred_df_lstm)
strata_lstm = build_error_strata_report(pred_df_lstm, "LSTM", spec_final.fold_name, spec_final.partition_name)

art_lstm = ModelArtifacts(
    model_name="LSTM",
    fold_name=spec_final.fold_name,
    partition_name=spec_final.partition_name,
    overall=overall_lstm,
    regression_by_horizon=reg_h_lstm,
    positive_by_horizon=pos_h_lstm,
    outbreak_by_horizon=out_h_lstm,
    outbreak_confusion_by_horizon=conf_h_lstm,
    anticipation_summary=anti_lstm,
    anticipation_event_level=event_lstm,
    error_strata=strata_lstm,
    predictions=pred_df_lstm,
    training_history=pd.DataFrame(),
    top_checkpoints=pd.DataFrame(),
    metrics_by_municipality=by_muni_lstm,
    metadata={"selected_checkpoint_path": str(LSTM_CKPT), "loaded_from_repo_checkpoint": True},
)
save_model_artifacts(spec_final, art_lstm)

print("\nLSTM — métricas globales:")
print(f"  MAE:                {art_lstm.overall.get('MAE', float('nan')):.4f}")
print(f"  RMSE:               {art_lstm.overall.get('RMSE', float('nan')):.4f}")
print(f"  Outbreak F1:        {art_lstm.overall.get('outbreak_model_f1', float('nan')):.4f}")
print(f"  Outbreak FAR:       {art_lstm.overall.get('outbreak_model_false_alarm_rate', float('nan')):.4f}")


## TFT

Carga el checkpoint del TFT y ejecuta predicción sobre el conjunto de
test. La función `to_prediction()` de `NegativeBinomialDistributionLoss`
deriva la media de la distribución (no muestrea), así que la inferencia
es completamente determinista.


In [ ]:
eval_start_idx = int(df_model.loc[df_model[YEAR_COL] == TEST_YEAR, TIME_COL].min())
training_ds_tft, eval_ds_tft = build_tft_datasets(
    train_df=spec_final.train_df,
    full_eval_df=spec_final.eval_df,
    prediction_start_idx=eval_start_idx,
)

eval_loader_tft = eval_ds_tft.to_dataloader(
    train=False,
    batch_size=TFT_BATCH_SIZE,
    num_workers=NUM_WORKERS,
)

tft_model = TemporalFusionTransformer.load_from_checkpoint(str(TFT_CKPT), map_location="cpu")
tft_model.eval()

pred_obj_tft = tft_model.predict(eval_loader_tft, return_y=True, return_index=True)
pred_df_tft_raw = build_prediction_frame_tft(pred_obj_tft)
pred_df_tft = prepare_predictions_for_evaluation(pred_df_tft_raw, source_eval_df=spec_final.eval_df)

overall_tft, reg_h_tft, pos_h_tft, out_h_tft, conf_h_tft, by_muni_tft = compute_overall_and_horizon_metrics(
    pred_df=pred_df_tft,
    model_name="TFT",
    fold_name=spec_final.fold_name,
    partition_name=spec_final.partition_name,
)

event_tft, anti_tft = build_outbreak_anticipation_report(pred_df_tft)
strata_tft = build_error_strata_report(pred_df_tft, "TFT", spec_final.fold_name, spec_final.partition_name)

art_tft = ModelArtifacts(
    model_name="TFT",
    fold_name=spec_final.fold_name,
    partition_name=spec_final.partition_name,
    overall=overall_tft,
    regression_by_horizon=reg_h_tft,
    positive_by_horizon=pos_h_tft,
    outbreak_by_horizon=out_h_tft,
    outbreak_confusion_by_horizon=conf_h_tft,
    anticipation_summary=anti_tft,
    anticipation_event_level=event_tft,
    error_strata=strata_tft,
    predictions=pred_df_tft,
    training_history=pd.DataFrame(),
    top_checkpoints=pd.DataFrame(),
    metrics_by_municipality=by_muni_tft,
    metadata={"selected_checkpoint_path": str(TFT_CKPT), "loaded_from_repo_checkpoint": True},
)
save_model_artifacts(spec_final, art_tft)

print("\nTFT — métricas globales:")
print(f"  MAE:                {art_tft.overall.get('MAE', float('nan')):.4f}")
print(f"  RMSE:               {art_tft.overall.get('RMSE', float('nan')):.4f}")
print(f"  Outbreak F1:        {art_tft.overall.get('outbreak_model_f1', float('nan')):.4f}")
print(f"  Outbreak FAR:       {art_tft.overall.get('outbreak_model_false_alarm_rate', float('nan')):.4f}")


## Resumen consolidado

Construye el reporte comparativo de los cuatro modelos y lo exporta a
Excel.


In [ ]:
artifacts_dict = {
    "NAIVE": art_naive,
    "ARIMA": art_arima,
    "LSTM": art_lstm,
    "TFT": art_tft,
}

build_fold_comparison_report(spec_final, artifacts_dict)
refresh_global_summary(RESULTS_DIR, partition_name="final_test")

# Tabla rápida en pantalla
rows = []
for model_name, art in artifacts_dict.items():
    rows.append({
        "model": model_name,
        "MAE": art.overall.get("MAE"),
        "RMSE": art.overall.get("RMSE"),
        "outbreak_F1": art.overall.get("outbreak_model_f1"),
        "outbreak_FAR": art.overall.get("outbreak_model_false_alarm_rate"),
        "positive_F1": art.overall.get("positive_model_f1"),
    })
summary_df = pd.DataFrame(rows)
print("\n========== RESUMEN ==========")
print(summary_df.to_string(index=False))
print(f"\nReportes completos en: {RESULTS_DIR}")
